In [1]:
import os
import subprocess
import time
import sys
import threading
import socket
from urllib.request import Request, urlopen
import re
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 -q
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 transformers torch sentencepiece accelerate pandas datasets "bitsandbytes>=0.40.0" -q


!pip install bcrypt

# Function to install packages
def install_packages():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit", "pyngrok", "pyjwt", "watchdog"])
print("Installing required packages...")
install_packages()

# Import after installation
from pyngrok import ngrok

# --- Create Streamlit Config for Dark Theme ---
os.makedirs(".streamlit", exist_ok=True)
config_toml = """
[theme]
base="dark"
primaryColor="#00F5FF"
backgroundColor="#041C32"
secondaryBackgroundColor="#06283D"
textColor="#FFFFFF"
font="sans serif"

[server]
headless = true
"""
with open(".streamlit/config.toml", "w") as f:
    f.write(config_toml)

print("Applied Neon Dark Theme configuration.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
Installing required packages...
Applied Neon Dark Theme configuration.


In [2]:
%%writefile db.py
# ---------------- DATABASE ----------------
import sqlite3
import bcrypt
import datetime
import time
import os


# Google Drive DB support

GDRIVE_DIR = "/content/drive/MyDrive/TextMorph"

import sys
if "google.colab" in sys.modules:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Could not mount drive: {e}")

# Database path selection
if os.path.exists("/content/drive/MyDrive"):
    if not os.path.exists(GDRIVE_DIR):
        os.makedirs(GDRIVE_DIR)
    DB_NAME = os.path.join(GDRIVE_DIR, "users.db")
else:
    DB_NAME = "users.db"


max_login_attempts = 3
lockout_time = 300  # 5 minutes in seconds


def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
    CREATE TABLE IF NOT EXISTS user_activity (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        action TEXT,
        language TEXT,
        created_at TEXT
    )
    """)
    # Locked Accounts Table
    c.execute("""
    CREATE TABLE IF NOT EXISTS locked_accounts (
        email TEXT PRIMARY KEY,
        locked_at TEXT
    )
    """)

    c.execute("""
    CREATE TABLE IF NOT EXISTS deleted_users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        username TEXT,
        deleted_at TEXT
    )
    """)



    # ---------- USER ROLE TABLE ----------
    c.execute("""
    CREATE TABLE IF NOT EXISTS user_roles(
        email TEXT PRIMARY KEY,
        role TEXT DEFAULT 'user'
    )
    """)

    # ---------- USER PROFILE ----------
    c.execute("""
    CREATE TABLE IF NOT EXISTS user_profiles(
        email TEXT PRIMARY KEY,
        avatar BLOB
    )
    """)

    # Users table
    c.execute('''
    CREATE TABLE IF NOT EXISTS users (
        email TEXT PRIMARY KEY,
        username TEXT,
        password BLOB,
        security_question TEXT,
        security_answer BLOB,
        created_at TEXT
    )
    ''')

    # Password History table
    c.execute('''
    CREATE TABLE IF NOT EXISTS password_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        password BLOB,
        set_at TEXT,
        FOREIGN KEY(email) REFERENCES users(email)
    )
    ''')

    # Login Attempts table
    c.execute('''
    CREATE TABLE IF NOT EXISTS login_attempts (
        email TEXT PRIMARY KEY,
        attempts INTEGER,
        last_attempt REAL
    )
    ''')

    # Feedback table
    c.execute("""
    CREATE TABLE IF NOT EXISTS feedback (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        original_text TEXT,
        generated_text TEXT,
        task_type TEXT,
        rating INTEGER,
        comments TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)


    # Activity History table
    c.execute('''CREATE TABLE IF NOT EXISTS activity_history
                (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  activity_type TEXT,
                  details TEXT,
                  output_text TEXT,
                  model_used TEXT,
                  language TEXT,
                  created_at TEXT)''')

    conn.commit()
    conn.close()


    # Ensure Admin Exists
    init_admin()

def promote_to_admin(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Insert if not exists, otherwise update
    c.execute("""
        INSERT INTO user_roles (email, role)
        VALUES (?, 'admin')
        ON CONFLICT(email) DO UPDATE SET role='admin'
    """, (email,))
    conn.commit()
    conn.close()


def delete_user(email):

    if email == "admin@textmorph.com":
        return False

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT email, username FROM users WHERE email=?", (email,))
    user = c.fetchone()

    if user:
        c.execute("""
        INSERT INTO deleted_users (email, username, deleted_at)
        VALUES (?, ?, ?)
        """, (user[0], user[1], _get_timestamp()))

    c.execute("DELETE FROM users WHERE email=?", (email,))
    c.execute("DELETE FROM password_history WHERE email=?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email=?", (email,))

    conn.commit()
    conn.close()

    return True

def save_profile_image(email, image_bytes):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
    INSERT OR REPLACE INTO user_profiles (email, avatar)
    VALUES (?,?)
    """, (email, image_bytes))

    conn.commit()
    conn.close()


def get_profile_image(email):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT avatar FROM user_profiles WHERE email=?", (email,))
    data = c.fetchone()

    conn.close()

    if data:
        return data[0]
    return None
def delete_profile_image(email):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
    UPDATE user_profiles
    SET avatar = NULL
    WHERE email = ?
    """, (email,))

    conn.commit()
    conn.close()
# Get locked accounts
def get_locked_accounts():

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT email, locked_at FROM locked_accounts")

    data = c.fetchall()
    conn.close()

    return data


# Unlock account
def unlock_account(email):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("DELETE FROM locked_accounts WHERE email=?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email=?", (email,))

    conn.commit()
    conn.close()


# Admin manually lock account
def lock_account(email):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
    INSERT OR REPLACE INTO locked_accounts(email, locked_at)
    VALUES (?,?)
    """, (email, _get_timestamp()))

    conn.commit()
    conn.close()

def get_username(email):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT username FROM users WHERE email=?", (email,))
    row = cursor.fetchone()
    conn.close()
    return row[0] if row else None

def init_admin():
    if not check_user_exists("admin@textmorph.com"):
        register_user("Admin", "admin@textmorph.com", "Admin@123Secure!", "Admin default question", "admin")
        print("Admin account created.")

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# --- Rate Limiting ---
def get_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    attempts, _ = get_login_attempts(email)
    new_attempts = attempts + 1
    now = time.time()

    c.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)",
              (email, new_attempts, now))
    conn.commit()
    conn.close()

def reset_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= max_login_attempts:
        if time.time() - last_attempt < lockout_time:
            return True, lockout_time - (time.time() - last_attempt)
        else:
            # Lockout expired
            reset_login_attempts(email)
    return False, 0

# --- User Management ---
def register_user(username, email, password, question, answer):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        hashed_password = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
        hashed_answer = bcrypt.hashpw(answer.encode(), bcrypt.gensalt())
        now = _get_timestamp()

        c.execute("""
        INSERT INTO users (email, username, password, security_question, security_answer, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (email, username, hashed_password, question, hashed_answer, now))
        # assign default role
        c.execute(
            "INSERT OR IGNORE INTO user_roles (email, role) VALUES (?, 'user')",
            (email,)
        )

        c.execute("""
        INSERT INTO password_history (email, password, set_at)
        VALUES (?, ?, ?)
        """, (email, hashed_password, now))

        conn.commit()
        return True

    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def get_security_question(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT security_question FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data[0] if data else None


def verify_security_answer(email, answer):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT security_answer FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()

    if data:
        stored_hash = data[0]
        return bcrypt.checkpw(answer.encode('utf-8'), stored_hash)

    return False

def authenticate_user(email, password):

    # Check if admin already locked account
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT email FROM locked_accounts WHERE email=?", (email,))
    locked = c.fetchone()

    conn.close()

    if locked:
        return "locked"

    # Check rate limit
    locked_rate, _ = is_rate_limited(email)
    if locked_rate:
        lock_account(email)   # ⭐ Add user to locked_accounts table
        return "locked"

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone()

    conn.close()

    if data:
        stored_hash = data[0]

        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            reset_login_attempts(email)
            return True

    increment_login_attempts(email)

    # If attempts reached limit → lock account
    attempts, _ = get_login_attempts(email)
    if attempts >= max_login_attempts:
        lock_account(email)   # ⭐ Add to locked_accounts

    return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
    history = c.fetchall()
    conn.close()

    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            return set_at
    return None

def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode('utf-8'), stored_hash):
            return True
    return False

def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data is not None

def update_password(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(new_password.encode('utf-8'), salt)
    now = _get_timestamp()

    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

    conn.commit()
    conn.close()

# --- Feedback System ---
def save_feedback(email, original_text, generated_text, task_type, rating, comments):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    now = _get_timestamp()

    c.execute("""
    INSERT INTO feedback
    (email, original_text, generated_text, task_type, rating, comments, created_at)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (email, original_text, generated_text, task_type, rating, comments, now))

    conn.commit()
    conn.close()

def get_all_feedback():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT id, email, task_type, rating, comments, created_at
        FROM feedback
        ORDER BY created_at DESC
    """)

    feedbacks = c.fetchall()
    conn.close()

    return feedbacks

# --- Activity History System ---
def log_activity(email, activity_type, details, output_text, model_used, language="English"):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()

    c.execute("""
    INSERT INTO activity_history
    (email, activity_type, details, output_text, model_used, language, created_at)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (email, activity_type, details, output_text, model_used, language, now))

    conn.commit()
    conn.close()

def get_user_activity(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
    SELECT activity_type, details, output_text, model_used, created_at
    FROM activity_history
    WHERE email = ? AND activity_type != 'Login'
    ORDER BY created_at DESC
    """, (email,))

    activities = c.fetchall()
    conn.close()
    return activities

# --- Admin Functions ---
def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, created_at FROM users")
    data = c.fetchall()
    conn.close()
    return data
def get_all_activity():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            email,
            activity_type,
            details,
            language,
            model_used,
            created_at,
            output_text
        FROM activity_history
        ORDER BY created_at DESC
    """)

    data = cursor.fetchall()
    conn.close()
    return data

Writing db.py


In [3]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }

Writing readability.py


In [4]:
%%writefile engine.py
import os
import torch
import streamlit as st
import nltk
from nltk.tokenize import sent_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

TRANSFORMERS_AVAILABLE = False
BNB_AVAILABLE = False
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
    TRANSFORMERS_AVAILABLE = True
    try:
        from transformers import BitsAndBytesConfig
        BNB_AVAILABLE = True
    except ImportError:
        pass
except ImportError as e:
    TRANSFORMERS_AVAILABLE = False

# ─── NLLB Translation Support ───
LANG_CODES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "Kannada": "kan_Knda",
    "Telugu": "tel_Telu",
    "Marathi": "mar_Deva",
    "Bengali": "ben_Beng",
    "Gujarati": "guj_Gujr",
    "Malayalam": "mal_Mlym",
    "Urdu": "urd_Arab",
    "Punjabi": "pan_Guru"
}

@st.cache_resource(show_spinner=False)
def load_translation_model():
    """Load NLLB-200 distilled model for multilanguage translation"""
    if not TRANSFORMERS_AVAILABLE:
        return None, None
    try:
        model_id = "facebook/nllb-200-distilled-600M"
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")
        return tokenizer, model
    except Exception as e:
        print(f"Warning: Could not load NLLB translation model: {e}")
        return None, None

def translate_text(text, source_lang="English", target_lang="English"):
    """Translate text between supported languages using NLLB-200"""
    if source_lang == target_lang:
        return text

    trans_tok, trans_model = load_translation_model()
    if trans_tok is None or trans_model is None:
        return text  # fallback: return untranslated

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    try:
        # Process in chunks to handle long texts
        sentences = sent_tokenize(text)
        chunks = []
        curr_chunk = []
        curr_len = 0
        for s in sentences:
            s_len = len(s.split())
            if curr_len + s_len > 200 and curr_chunk:
                chunks.append(" ".join(curr_chunk))
                curr_chunk = [s]
                curr_len = s_len
            else:
                curr_chunk.append(s)
                curr_len += s_len
        if curr_chunk:
            chunks.append(" ".join(curr_chunk))

        translated_parts = []
        for chunk in chunks:
            trans_tok.src_lang = src_code
            inputs = trans_tok(chunk, return_tensors="pt", max_length=512, truncation=True).to(trans_model.device)
            tgt_token_id = trans_tok.convert_tokens_to_ids(tgt_code)
            with torch.no_grad():
                outputs = trans_model.generate(**inputs, forced_bos_token_id=tgt_token_id, max_length=384, use_cache=True)
            translated_parts.append(trans_tok.decode(outputs[0], skip_special_tokens=True))

        return " ".join(translated_parts)
    except Exception as e:
        print(f"Translation error: {e}")
        return text


@st.cache_resource(show_spinner=False)
def load_summarization_models(quantization_level="4-bit"):
    """Load summarization models with 4-bit quantization by default for speed"""
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models

    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

    # 1. BART MODEL
    try:
        models['bart'] = {
            'tokenizer': AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6", **kwargs)
        }
    except Exception as e:
        print(f"BART load failed: {e}")
        models['bart'] = None

    # 2. PEGASUS MODEL
    try:
        models['pegasus'] = {
            'tokenizer': AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-cnn_dailymail", **kwargs)
        }
    except Exception as e:
        print(f"Pegasus load failed: {e}")
        models['pegasus'] = None

    # 3. FLAN-T5 MODEL
    try:
        t5_models_to_try = ["google/flan-t5-base", "google/flan-t5-small"]
        t5_loaded = False
        for t5_model in t5_models_to_try:
            try:
                models['flan-t5'] = {
                    'tokenizer': AutoTokenizer.from_pretrained(t5_model),
                    'model': AutoModelForSeq2SeqLM.from_pretrained(t5_model, **kwargs)
                }
                t5_loaded = True
                break
            except Exception:
                continue
        if not t5_loaded:
            models['flan-t5'] = None
    except Exception as e:
        print(f"FLAN-T5 load failed: {e}")
        models['flan-t5'] = None

    return models

@st.cache_resource(show_spinner=False)
def load_paraphrase_models(quantization_level="4-bit"):
    """Load paraphrase models with 4-bit quantization by default"""
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models

    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

    try:
        try:
            models['flan_t5'] = {
                'tokenizer': AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws", **kwargs)
            }
        except Exception:
            try:
                models['flan_t5'] = {
                    'tokenizer': AutoTokenizer.from_pretrained("google/flan-t5-small"),
                    'model': AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", **kwargs)
                }
            except Exception:
                models['flan_t5'] = None

        try:
            models['bart'] = {
                'tokenizer': AutoTokenizer.from_pretrained("eugenesiow/bart-paraphrase"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("eugenesiow/bart-paraphrase", **kwargs)
            }
        except Exception:
            models['bart'] = None

        return models
    except Exception:
        return {}


def _detect_hallucination(original_text, generated_text):
    """Post-generation quality check to detect hallucinated output"""
    gen_words = generated_text.split()
    orig_words = set(original_text.lower().split())

    if len(gen_words) < 3:
        return True

    # Check for excessive repetition (same word appearing > 50% of output)
    from collections import Counter
    word_counts = Counter(w.lower().strip(".,!?();:'\"") for w in gen_words)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    if most_common_count > len(gen_words) * 0.5 and len(gen_words) > 20:
        return True

    # Check if output has too many words not in original (hallucination signal)
    # Only flag if >85% of words are completely novel AND output is long
    gen_clean = [w.lower().strip(".,!?();:'\"") for w in gen_words]
    novel_words = [w for w in gen_clean if w not in orig_words and len(w) > 3]
    if len(novel_words) > len(gen_words) * 0.85 and len(gen_words) > 30:
        return True

    return False


def simple_text_summarization(text, summary_length):
    """Simple extractive text summarization fallback"""
    try:
        sentences = sent_tokenize(text)
        if len(sentences) <= 2:
            return text[:100] + "..." if len(text) > 100 else text

        if summary_length == "Short":
            return " ".join(sentences[:max(1, len(sentences) // 4)])
        elif summary_length == "Medium":
            return " ".join(sentences[:max(2, len(sentences) // 2)])
        else:
            return " ".join(sentences[:max(3, int(len(sentences) * 0.75))])
    except:
        return text[:150] + "..." if len(text) > 150 else text


def local_summarize(text, summary_length, model_type, models_dict, target_lang="English"):
    """Summarization with anti-hallucination guardrails and multilanguage support"""
    model_key = model_type.lower()

    if (model_key not in models_dict or models_dict[model_key] is None):
        st.warning(f"⚠️ {model_type} model not available. Using fallback method.")
        result = simple_text_summarization(text, summary_length)
        if target_lang != "English":
            result = translate_text(result, "English", target_lang)
        return result

    model_info = models_dict[model_key]
    tokenizer = model_info['tokenizer']
    model = model_info['model']

    input_word_count = len(text.split())
    input_length = len(tokenizer.encode(text))

    # Adaptive length config with WIDER gaps between Short/Medium/Long
    is_long_doc = input_word_count >= 1000

    if is_long_doc:
        length_config = {
            "Short":  {"max_length": min(300, max(150, input_length // 4)),   "min_length": min(100, max(50, input_length // 6))},
            "Medium": {"max_length": min(700, max(400, int(input_length * 0.5))), "min_length": min(350, max(200, input_length // 3))},
            "Long":   {"max_length": min(1500, max(800, int(input_length * 0.85))), "min_length": min(700, max(500, int(input_length * 0.6)))}
        }
    else:
        safe_max = max(60, int(input_length * 0.95))
        length_config = {
            "Short":  {"max_length": min(60, max(20, input_length // 4)),   "min_length": min(10, max(5, input_length // 6))},
            "Medium": {"max_length": min(150, max(40, input_length // 2)),  "min_length": min(25, max(12, input_length // 4))},
            "Long":   {"max_length": min(safe_max, max(80, int(input_length * 0.9))), "min_length": min(50, max(25, input_length // 2))}
        }

    config = length_config.get(summary_length, length_config["Medium"])

    # Ensure min never exceeds max
    config["min_length"] = min(config["min_length"], config["max_length"] - 5)
    config["min_length"] = max(config["min_length"], 5)

    # FLAN-T5: length-specific prompts so Short/Medium/Long produce different results
    if model_key == 'flan-t5':
        if summary_length == "Short":
            prompt = f"Write a brief 2-3 sentence summary of the following text: {text}"
        elif summary_length == "Medium":
            prompt = f"Write a detailed summary of the following text, covering the main points: {text}"
        else:
            prompt = f"Write a comprehensive and thorough summary of the following text, covering all key points and important details: {text}"
    else:
        prompt = text

    try:
        with st.spinner(f"🧠 {model_type} generating summary..."):
            device = next(model.parameters()).device
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
                padding=True
            ).to(device)

            gen_kwargs = {
                "max_new_tokens": config["max_length"],
                "min_new_tokens": config["min_length"],
                "num_beams": 2,
                "length_penalty": {"Short": 0.6, "Medium": 1.0, "Long": 1.8}.get(summary_length, 1.0),
                "no_repeat_ngram_size": 3,
                "early_stopping": True,
                "use_cache": True,
                "repetition_penalty": 1.5,
            }

            with torch.no_grad():
                outputs = model.generate(**inputs, **gen_kwargs)

            summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Post-generation hallucination check
            if _detect_hallucination(text, summary):
                summary = simple_text_summarization(text, summary_length)

            if not summary.strip():
                summary = simple_text_summarization(text, summary_length)

            # For long documents, if summary is too short, pad with extractive sentences
            if is_long_doc and len(summary.split()) < 400:
                extractive_addition = simple_text_summarization(text, "Long")
                summary = summary + "\n\n" + extractive_addition

            # Translate if needed
            if target_lang != "English":
                with st.spinner(f"🌐 Translating to {target_lang}..."):
                    summary = translate_text(summary, "English", target_lang)

            return summary
    except Exception as e:
        st.error(f"❌ {model_type} AI MODEL ERROR: {str(e)}")
        result = simple_text_summarization(text, summary_length)
        if target_lang != "English":
            result = translate_text(result, "English", target_lang)
        return result


def apply_fallback_paraphrasing(text, complexity):
    words = text.split()
    if len(words) <= 3:
        return text

    substitutions = {
        "Beginner": {
            "utilize": "use", "facilitate": "help", "fundamental": "basic",
            "however": "but", "moreover": "also", "subsequently": "then",
            "very": "quite", "important": "key"
        },
        "Intermediate": {
            "use": "utilize", "help": "assist", "basic": "fundamental",
            "but": "however", "also": "furthermore", "then": "subsequently",
            "important": "significant", "good": "effective"
        },
        "Advanced": {
            "use": "leverage", "help": "facilitate", "basic": "foundational",
            "but": "nevertheless", "also": "moreover", "then": "thereafter",
            "show": "demonstrate", "important": "paramount", "good": "optimal"
        },
        "Expert": {
            "use": "employ", "help": "ameliorate", "basic": "rudimentary",
            "show": "elucidate", "make": "synthesize", "important": "critical", "good": "superior"
        }
    }
    sub_dict = substitutions.get(complexity, substitutions["Intermediate"])
    paraphrased_words = []
    for word in words:
        clean_word = word.strip(".,!?();:'\"").lower()
        if clean_word in sub_dict:
            new_word = sub_dict[clean_word]
            if word[0].isupper():
                new_word = new_word.capitalize()
            replaced = word.lower().replace(clean_word, new_word)
            if word[0].isupper():
                replaced = replaced.capitalize()
            paraphrased_words.append(replaced)
        else:
            paraphrased_words.append(word)
    return " ".join(paraphrased_words)


def paraphrase_with_model(text, complexity, style, model_type, models_dict, target_lang="English"):
    """Paraphrase text using specified model with multilanguage support"""
    model_key = model_type.lower().replace('-', '_')
    try:
        model_info = models_dict.get(model_key)
        if model_info is None:
            result = apply_fallback_paraphrasing(text, complexity)
            if target_lang != "English":
                result = translate_text(result, "English", target_lang)
            return result

        tokenizer = model_info['tokenizer']
        model = model_info['model']
        device = next(model.parameters()).device

        # Split text into smaller chunks for better paraphrasing quality
        sentences = sent_tokenize(text)
        chunks = []
        curr = []
        curr_len = 0
        for s in sentences:
            slen = len(s.split())
            if curr_len + slen > 80 and curr:
                chunks.append(" ".join(curr))
                curr = [s]
                curr_len = slen
            else:
                curr.append(s)
                curr_len += slen
        if curr:
            chunks.append(" ".join(curr))

        paraphrased_chunks = []
        for chunk in chunks:
            chunk_token_count = len(tokenizer.encode(chunk))

            if model_key == 'flan_t5':
                prompt = f"paraphrase the following text using different words and sentence structure: {chunk} </s>"
            else:
                prompt = f"paraphrase: {chunk}"

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding="max_length"
            ).to(device)

            # Scale output to match input length — fast greedy decoding
            max_out = max(150, int(chunk_token_count * 1.5))
            min_out = max(10, int(chunk_token_count * 0.6))

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_out,
                    min_new_tokens=min_out,
                    num_beams=1,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.8,
                    use_cache=True
                )

            paraphrased = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if len(paraphrased.strip()) > 10:
                paraphrased_chunks.append(paraphrased)
            else:
                paraphrased_chunks.append(chunk)

        final_paraphrase = " ".join(paraphrased_chunks)
        if not final_paraphrase.strip():
            final_paraphrase = apply_fallback_paraphrasing(text, complexity)

        # Translate if needed
        if target_lang != "English":
            with st.spinner(f"🌐 Translating to {target_lang}..."):
                final_paraphrase = translate_text(final_paraphrase, "English", target_lang)

        return final_paraphrase
    except Exception as e:
        st.error(f"❌ Paraphrasing Engine Error ({model_type}): {str(e)}")
        result = apply_fallback_paraphrasing(text, complexity)
        if target_lang != "English":
            result = translate_text(result, "English", target_lang)
        return result

Writing engine.py


In [5]:
%%writefile app.py
import streamlit as st
import jwt
import sqlite3
import re
import os
import bcrypt
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import secrets
import time
import hmac
import hashlib
import struct
import db
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import PyPDF2
import pandas as pd
import engine
import random
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import base64
from io import BytesIO
import datetime
import pytz

# ---------------- CONFIG ----------------
ALGORITHM = "HS256"

st.set_page_config(
    page_title="Infosys Springboard",
    page_icon="🎓",
    layout="wide"
)

# Initialize DB
if 'db_initialized' not in st.session_state:
    db.init_db()
    st.session_state['db_initialized'] = True
# Configuration
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SECRET_KEY = os.getenv("JWT_SECRET", "super-secret-key-change-this")
EMAIL_ADDRESS ="Infosysteam91@gmail.com"
OTP_EXPIRY_MINUTES = 10

# Supported languages for multilanguage summarization/paraphrasing
SUPPORTED_LANGUAGES = [
    "English",
    "Hindi",
    "Tamil",
    "Kannada",
    "Telugu",
    "Marathi",
    "Bengali",
    "Gujarati",
    "Malayalam",
    "Urdu",
    "Punjabi"
]

# --- Load ALL Models Eagerly At Startup (before login) ---
if 'summarization_models' not in st.session_state:
    with st.spinner("🚀 Loading AI models (4-bit quantized for speed)..."):
        st.session_state.summarization_models = engine.load_summarization_models()
        st.session_state.paraphrase_models = engine.load_paraphrase_models()
        engine.load_translation_model()


# ============================================================
#  DESIGN SYSTEM  — single source of truth for all colours /
#  spacing / typography used throughout the app.
# ============================================================
NEON   = "#00C8FF"   # electric cyan
NEON2  = "#0099CC"   # deep cyan
BG1    = "#0A0F1E"   # midnight navy
BG2    = "#0D1526"   # deep navy
GLASS  = "rgba(0,200,255,0.06)"
BORDER = "rgba(0,200,255,0.28)"
RED    = "#FF4C6A"
YELLOW = "#00E5FF"   # pure gold
TEXT   = "#E8F4FD"   # cool white
MUTED  = "#6B8BA4"   # slate blue-grey

# ── Global CSS ────────────────────────────────────────────────

st.markdown(f"""
<style>
/* ── Google Fonts ── */
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap');

/* ── Base ── */
html, body, [data-testid="stAppViewContainer"] {{
    background: #0A0F1E !important;
    color: {TEXT};
    font-family: 'DM Sans', sans-serif;
}}
/* Show Streamlit header (with 3-dot menu) */
header {{ visibility: visible !important; }}

/* Hide only footer (optional) */
footer {{ visibility: hidden; }}

/* Hide only "Made with Streamlit" menu */
#MainMenu {{ visibility: visible; }}  /* keep visible if you want full menu */


/* ── Headings ── */
h1 {{ font-family:'Syne',sans-serif; font-size:2rem; font-weight:800;
       color:{NEON}; letter-spacing:-0.5px; margin-bottom:0.25rem; }}
h2 {{ font-family:'Syne',sans-serif; font-size:1.4rem; font-weight:700;
       color:{NEON}; }}
h3 {{ font-family:'Syne',sans-serif; font-size:1.1rem; font-weight:600;
       color:#B8E4F9; }}
/* ── Inputs ── */
.stTextInput > div > div > input,
.stTextArea  > div > div > textarea {{
    background: rgba(10,20,40,0.90) !important;
    border: 1px solid {BORDER} !important;
    border-radius: 10px !important;
    color: {TEXT} !important;
    font-size: 0.95rem !important;
    padding: 10px 14px !important;
    transition: border-color .2s;
}}
.stTextInput > div > div > input:focus,
.stTextArea  > div > div > textarea:focus {{
    border-color: {NEON} !important;
    box-shadow: 0 0 0 2px rgba(0,200,255,.15) !important;
}}

/* ── Select boxes ── */
.stSelectbox > div > div {{
    background: rgba(10,20,40,0.90) !important;
    border: 1px solid {BORDER} !important;
    border-radius: 10px !important;
    color: {TEXT} !important;
}}

/* ── Primary button ── */
.stButton > button[kind="primary"],
.stButton > button {{
    background: linear-gradient(90deg, {NEON}, {NEON2}) !important;
    color: #0A0F1E !important;
    font-weight: 700 !important;
    font-size: 0.9rem !important;
    border: none !important;
    border-radius: 50px !important;
    padding: 10px 28px !important;
    box-shadow: 0 0 18px rgba(0,200,255,.45) !important;
    transition: transform .15s, box-shadow .15s !important;
    white-space: nowrap !important;
}}
.stButton > button:hover {{
    transform: translateY(-2px) !important;
    box-shadow: 0 0 30px rgba(0,200,255,.75) !important;
}}
/* secondary-style override */
button[kind="secondary"] {{
    background: transparent !important;
    color: {NEON} !important;
    border: 1px solid {NEON} !important;
    border-radius: 50px !important;
    box-shadow: none !important;
}}
button[kind="secondary"]:hover {{
    background: rgba(0,200,255,.12) !important;
}}

/* ── Cards ── */
.tm-card {{
    background: rgba(13,24,46,0.95);
    backdrop-filter: blur(14px);
    border: 1px solid {BORDER};
    border-radius: 16px;
    padding: 30px 32px !important;
    border-radius: 20px;
    box-shadow: 0 0 22px rgba(0,200,255,.12);
    transition: box-shadow .25s, border-color .25s;
    margin-bottom: 20px;
}}
.tm-card:hover {{
    border-color: {NEON};
    box-shadow: 0 0 35px rgba(0,200,255,.30);
}}

/* ── Tool-icon cards (home page grid) ── */
.tool-card {{
    background: linear-gradient(145deg, #0F1B2D, #0C1828);
    border: 1px solid {BORDER};
    border-radius: 18px;
    padding: 28px 24px;
    text-align: center;
    cursor: pointer;
    transition: all .25s;
    height: 100%;
}}
.tool-card:hover {{
    border-color: {NEON};
    transform: translateY(-4px);
    box-shadow: 0 8px 32px rgba(0,200,255,.30);
}}
.tool-card .tc-icon {{ font-size: 2.4rem; margin-bottom: 14px; }}
.tool-card .tc-title {{
    font-family: 'Syne', sans-serif;
    font-size: 1.05rem; font-weight: 700;
    color: {NEON}; margin-bottom: 6px;
}}
.tool-card .tc-desc {{ font-size: 0.82rem; color: {MUTED}; line-height: 1.5; }}

/* ── Badges / chips ── */
.badge {{
    display: inline-block;
    font-size: 0.72rem; font-weight: 600;
    padding: 3px 11px; border-radius: 50px;
    border: 1px solid currentColor;
    margin: 2px;
    font-family: 'DM Sans', sans-serif;
    letter-spacing: .3px;
}}
.badge-cyan  {{ color:{NEON};         background:rgba(0,200,255,.12); }}
.badge-amber {{ color:{NEON};         background:rgba(0,200,255,.12); }}
.badge-teal  {{ color:#2DD4BF;        background:rgba(45,212,191,.10); }}
.badge-green {{ color:#34D399;        background:rgba(52,211,153,.10); }}
.badge-red   {{ color:{RED};          background:rgba(255,76,106,.1); }}
.badge-gold  {{ color:{YELLOW};       background:rgba(252,211,77,.1); }}

/* ── Result panels ── */
.result-panel {{
    background: #0D1526;
    border-left: 3px solid {NEON};
    border-radius: 0 12px 12px 0;
    padding: 20px 24px;
    font-size: 0.95rem;
    line-height: 1.7;
    color: {TEXT};
}}
.result-panel-success {{
    background: #0D1A2E;
    border-left: 3px solid #34D399;
    border-radius: 0 12px 12px 0;
    padding: 20px 24px;
    font-size: 0.95rem;
    line-height: 1.7;
    color: {TEXT};
}}

/* ── Star rating ── */
.stars {{ font-size:1.4rem; letter-spacing:2px; }}

/* ── History rows ── */
.hist-row {{
    display: flex; align-items: flex-start; gap: 16px;
    background: {GLASS};
    border: 1px solid {BORDER};
    border-radius: 14px;
    padding: 18px 22px;
    margin-bottom: 12px;
    transition: border-color .2s;
}}
.hist-row:hover {{ border-color: {NEON}; }}
.hist-icon {{ font-size:1.6rem; flex-shrink:0; padding-top:2px; }}
.hist-meta {{ flex:1; min-width:0; }}
.hist-meta h4 {{ margin:0 0 4px; font-size:.95rem; font-weight:600;
                  color:{NEON}; font-family:'Syne',sans-serif; }}
.hist-meta p  {{ margin:0; font-size:.82rem; color:{MUTED}; }}
.hist-time  {{ font-size:.75rem; color:{MUTED}; white-space:nowrap; }}

/* ── Auth container ── */
.auth-wrap {{
    max-width: 520px;
    margin: 30px auto 0;
}}
.auth-logo {{
    font-family:'Syne',sans-serif;
    font-size:1.7rem; font-weight:800;
    color:{NEON};
    text-align:center;
    letter-spacing:-1px;
    margin-bottom: 6px;
}}
.auth-sub {{
    text-align:center; color:{MUTED};
    font-size:.85rem; margin-bottom:32px;
}}

/* ── Metric cards ── */
[data-testid="stMetric"] {{
    background: rgba(13,24,46,0.95);
    border: 1px solid {BORDER};
    border-radius: 14px;
    padding: 16px 20px !important;
}}
[data-testid="stMetricLabel"] {{ color:{MUTED} !important; font-size:.8rem !important; }}
[data-testid="stMetricValue"] {{
    color:{NEON} !important;
    font-family:'Syne',sans-serif !important;
    font-weight:700 !important;
}}

/* ── Tables ── */
[data-testid="stDataFrame"] table {{ background: transparent !important; }}
[data-testid="stDataFrame"] th {{
    background: rgba(0,200,255,.12) !important;
    color: {NEON} !important;
    font-size:.8rem; text-transform:uppercase; letter-spacing:.5px;
    border-bottom: 1px solid {BORDER} !important;
}}
[data-testid="stDataFrame"] td {{
    color: {TEXT} !important;
    border-bottom: 1px solid rgba(0,200,255,.07) !important;
    font-size:.85rem;
}}

/* ── Tabs ── */
.stTabs [data-baseweb="tab-list"] {{
    gap: 6px; background: transparent;
    border-bottom: 1px solid {BORDER};
}}
.stTabs [data-baseweb="tab"] {{
    border-radius: 10px 10px 0 0;
    color: {MUTED}; background: transparent;
    font-size:.88rem; font-weight:500;
    padding: 8px 20px;
}}
.stTabs [aria-selected="true"] {{
    background: rgba(0,200,255,.10) !important;
    color: {NEON} !important;
    border-bottom: 2px solid {NEON} !important;
}}

/* ── Expanders ── */
[data-testid="stExpander"] {{
    background: rgba(13,24,46,0.95);
    border: 1px solid {BORDER};
    border-radius: 12px;
}}
[data-testid="stExpander"] summary {{
    font-size:.9rem; font-weight:600; color:{NEON};
}}

/* ── Scrollbar ── */
::-webkit-scrollbar {{ width:5px; }}
::-webkit-scrollbar-track {{ background: {BG1}; }}
::-webkit-scrollbar-thumb {{ background:{BORDER}; border-radius:6px; }}

/* ── Page divider ── */
.section-divider {{
    border: none;
    border-top: 1px solid {BORDER};
    margin: 28px 0;
}}

/* ── Feedback review card ── */
.review-card {{
    background: rgba(14,14,14,.85);
    border: 1px solid {BORDER};
    border-radius: 14px;
    padding: 20px 24px;
    margin-bottom: 14px;
    transition: border-color .2s;
}}
.review-card:hover {{ border-color: {NEON}; }}
.review-header {{
    display:flex; justify-content:space-between; align-items:center;
    margin-bottom:10px;
}}
.review-task {{
    font-family:'Syne',sans-serif; font-weight:700;
    font-size:.95rem; color:{NEON};
}}
.review-meta {{ font-size:.78rem; color:{MUTED}; }}
.review-comment {{ font-size:.88rem; color:{TEXT}; line-height:1.6; margin-top:8px; }}


/* ── Page header strip ── */
.page-header {{
    display:flex; align-items:center; gap:14px;
    margin-bottom:28px;
}}
.page-header .ph-icon {{ font-size:2rem; }}
.page-header .ph-title {{
    font-family:'Syne',sans-serif; font-weight:800;
    font-size:1.6rem; color:{NEON};
    margin:0; line-height:1;
}}
.page-header .ph-sub {{
    font-size:.82rem; color:{MUTED}; margin:4px 0 0;
}}

/* ── Back-to-home button ── */
button[kind="secondary"],
div[data-testid="stButton"] > button[key^="back_"] {{
    background: transparent !important;
    color: {NEON} !important;
    border: 1px solid {BORDER} !important;
    border-radius: 50px !important;
    font-size: 0.82rem !important;
    padding: 6px 16px !important;
    box-shadow: none !important;
    font-weight: 600 !important;
}}
div[data-testid="stButton"] > button[key^="back_"]:hover {{
    background: rgba(0,200,255,.10) !important;
    border-color: {NEON} !important;
}}

/* ── Password strength bar ── */
.pw-bar-wrap {{
    height:6px; border-radius:4px;
    background:rgba(255,255,255,.06);
    overflow:hidden; margin-top:6px;
}}
.pw-bar {{ height:100%; border-radius:4px; transition:width .3s; }}

/* ── Plotly transparent bg ── */
.js-plotly-plot .plotly {{ background:transparent !important; }}

/* ── Admin stat card ── */
.admin-stat {{
    background:rgba(13,24,46,0.95);
    border:1px solid {BORDER};
    border-radius:16px;
    padding:24px 20px;
    text-align:center;
}}
.admin-stat .as-num {{
    font-family:'Syne',sans-serif; font-size:2rem;
    font-weight:800; color:{NEON};
}}
.admin-stat .as-label {{ font-size:.82rem; color:{MUTED}; margin-top:4px; }}

/* ── Main content area ── */


/* ── Column gap fix ── */
[data-testid="stHorizontalBlock"] {{
    gap: 1.25rem !important;
    align-items: stretch !important;
}}

/* ── Form element vertical rhythm ── */
.stTextInput, .stSelectbox, .stTextArea {{
    margin-bottom: 0.5rem !important;
}}
.stTextInput label, .stSelectbox label, .stTextArea label {{
    color: {MUTED} !important;
    font-size: 0.82rem !important;
    font-weight: 500 !important;
    margin-bottom: 4px !important;
}}
/* ── Card inner spacing consistency ── */
.tm-card > *:last-child {{ margin-bottom: 0 !important; }}

/* ── Page header bottom spacing ── */
.page-header {{ margin-bottom: 24px !important; }}

/* ── Metric column alignment ── */
[data-testid="stMetric"] {{
    height: 100% !important;
}}

/* ── Expander padding ── */
[data-testid="stExpander"] > div > div {{
    padding: 12px 16px !important;
}}

/* ── Button consistent height ── */
.stButton > button {{
    min-height: 42px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
}}

/* ── Download button style ── */
[data-testid="stDownloadButton"] > button {{
    background: transparent !important;
    color: {NEON} !important;
    border: 1px solid {BORDER} !important;
    border-radius: 50px !important;
    box-shadow: none !important;
    font-size: 0.85rem !important;
    padding: 8px 20px !important;
}}
[data-testid="stDownloadButton"] > button:hover {{
    background: rgba(0,200,255,.12) !important;
    border-color: {NEON} !important;
}}

/* ── Radio / checkbox labels ── */
.stRadio label, .stCheckbox label {{
    color: {TEXT} !important;
    font-size: 0.88rem !important;
}}
.stRadio [data-testid="stWidgetLabel"] {{
    color: {MUTED} !important;
    font-size: 0.82rem !important;
}}

/* ── File uploader ── */
[data-testid="stFileUploader"] {{
    background: rgba(28,18,5,.6) !important;
    border: 1px dashed {BORDER} !important;
    border-radius: 12px !important;
    padding: 12px !important;
}}
[data-testid="stFileUploader"] label {{
    color: {MUTED} !important;
}}

/* ── Spinner ── */
[data-testid="stSpinner"] {{
    color: {NEON} !important;
}}

/* ── st.info / st.success / st.error alignment ── */
[data-testid="stAlert"] {{
    border-radius: 10px !important;
    font-size: 0.88rem !important;
}}

/* Sidebar */
[data-testid="stSidebar"] {{
    background: #0D1526 !important;
}}
[data-testid="stSidebar"] div[data-testid="stButton"] > button {{
    min-height: 38px !important;
    padding: 6px 10px !important;
    font-size: 0.8rem !important;
    border-radius: 6px !important;
    box-shadow: none !important;
}}
</style>
""", unsafe_allow_html=True)




# ---------------- JWT ----------------
def create_token(data):
    data["exp"] = datetime.datetime.utcnow() + datetime.timedelta(hours=1)
    return jwt.encode(data, SECRET_KEY, algorithm=ALGORITHM)

# ---------------- VALIDATION ----------------
def get_relative_time(date_str):
    if not date_str: return "some time ago"
    try:
        past = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
        diff = datetime.datetime.utcnow() - past
        days = diff.days
        if days > 365: return f"{days // 365} years ago"
        elif days > 30: return f"{days // 30} months ago"
        elif days > 0: return f"{days} days ago"
        else: return "recently"
    except: return date_str
def valid_email(email):
    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
    return re.match(pattern, email)

def valid_password(password):
    pattern = r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$'
    return re.match(pattern, password)

def password_strength(password):
    score = 0
    if len(password) >= 8: score += 1
    if re.search(r'[A-Z]', password): score += 1
    if re.search(r'[a-z]', password): score += 1
    if re.search(r'\\d', password): score += 1
    if re.search(r'[@$!%*?&]', password): score += 1
    return score
# --- Security Logic ---

def generate_otp():
    secret = secrets.token_bytes(20)
    counter = int(time.time())
    msg = struct.pack(">Q", counter)
    hmac_hash = hmac.new(secret, msg, hashlib.sha1).digest()
    offset = hmac_hash[19] & 0xf
    code = ((hmac_hash[offset] & 0x7f) << 24 | (hmac_hash[offset + 1] & 0xff) << 16 | (hmac_hash[offset + 2] & 0xff) << 8 | (hmac_hash[offset + 3] & 0xff))
    otp = code % 1000000
    return f"{otp:06d}"

def create_otp_token(otp, email):
    otp_hash = bcrypt.hashpw(otp.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
    payload = {
        'otp_hash': otp_hash,
        'sub': email,
        'type': 'password_reset',
        'iat': datetime.datetime.utcnow(),
        'exp': datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm='HS256')

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
        if payload.get('sub') != email: return False, "Token mismatch"
        if bcrypt.checkpw(input_otp.encode('utf-8'), payload['otp_hash'].encode('utf-8')):
            return True, "Valid"
        return False, "Invalid OTP"
    except Exception as e:
        return False, str(e)

def send_email(to_email, otp, app_pass):
    msg = MIMEMultipart()
    msg['From'] = f"Infosys Springboard <{EMAIL_ADDRESS}>"
    msg['To'] = to_email
    msg['Subject'] = "🔐 Infosys Springboard - Password Reset OTP"

    body = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <style>
            .container {{
                font-family: Arial, sans-serif;
                background-color: #0e1117;
                padding: 40px;
                text-align: center;
                color: #ffffff;
            }}
            .card {{
                background-color: #1f2937;
                border-radius: 12px;
                padding: 30px;
                max-width: 500px;
                margin: auto;
                border: 1px solid #00F5FF;
            }}
            .otp-box {{
                font-size: 32px;
                font-weight: bold;
                letter-spacing: 6px;
                padding: 20px;
                margin: 20px 0;
                background: #00F5FF;
                color: black;
                border-radius: 8px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="card">
                <h2>Infosys Springboard Security</h2>
                <p>Use this OTP  for your Account {to_email}</p>
                <div class="otp-box">{otp}</div>
                <p>Valid for {OTP_EXPIRY_MINUTES} minutes</p>
            </div>
        </div>
    </body>
    </html>
    """

    msg.attach(MIMEText(body, 'html'))

    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(EMAIL_ADDRESS, app_pass if app_pass else EMAIL_PASSWORD)
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        s.quit()
        return True, "Sent"
    except Exception as e:
        return False, str(e)

# ---------------- SESSION ----------------

if "page" not in st.session_state:
    st.session_state.page = "login"

if "token" not in st.session_state:
    st.session_state.token = None


def switch_page(page):
    st.session_state['page'] = page
    st.rerun()

def logout():
    st.session_state['user'] = None
    st.session_state['page'] = 'login'
    st.rerun()

def create_gauge(value, title, min_val, max_val, color):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={"text": title},
        gauge={
            "axis": {"range": [min_val, max_val]},
            "bar": {"color": color},
            "steps": [
                {"range": [min_val, (min_val+max_val)/3], "color": "#1f2937"},
                {"range": [(min_val+max_val)/3, (min_val+max_val)*2/3], "color": "#374151"},
                {"range": [(min_val+max_val)*2/3, max_val], "color": "#4b5563"},
            ],
        }
    ))
    fig.update_layout(height=250, margin=dict(l=10, r=10, t=40, b=10))
    return fig

def get_greeting():
    ist = pytz.timezone("Asia/Kolkata")
    now = datetime.datetime.now(ist)   # ✅ get full datetime once
    hour = now.hour

    if 5 <= hour < 12:
        return "Good Morning.. 🌅"
    elif 12 <= hour < 17:
        return "Good Afternoon.. ☀️"
    elif 17 <= hour < 21:
        return "Good Evening.. 🌇"
    else:
        return "Good Night.. 🌙"
def readability_page():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_profile"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    user = st.session_state.get('user')

    if not user:
        switch_page('login')
        st.stop()

    st.title("📖 Text Readability Analyzer")

    # Input Method: Text or File
    tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File (TXT/PDF)"])
    text_input = ""

    with tab1:
        raw_text = st.text_area("Enter text to analyze (min 50 chars):", height=200)
        if raw_text: text_input = raw_text

    with tab2:
        uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
        if uploaded_file:
            try:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text = ""
                    for page in reader.pages:
                        text += page.extract_text() + "\n"
                    text_input = text
                    st.info(f"✅ Loaded {len(reader.pages)} pages from PDF.")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.info(f"✅ Loaded TXT file: {uploaded_file.name}")
            except Exception as e:
                st.error(f"Error reading file: {e}")

    if st.button("Analyze Readability", type="primary"):
        if len(text_input) < 50:
            st.error("Text is too short (min 50 chars). Please enter more text or upload a valid file.")
        else:
            import readability
            with st.spinner("Calculating advanced metrics..."):
                analyzer = readability.ReadabilityAnalyzer(text_input)
                score = analyzer.get_all_metrics()

            # --- Results Dashboard ---
            st.markdown("---")
            st.subheader("📊 Analysis Results")

            # 1. Overall Grade (Average)
            avg_grade = (score['Flesch-Kincaid Grade'] + score['Gunning Fog'] + score['SMOG Index'] + score['Coleman-Liau']) / 4

            # Determine Level
            if avg_grade <= 6: level, color = "Beginner (Elementary)", "#28a745"
            elif avg_grade <= 10: level, color = "Intermediate (Middle School)", "#17a2b8"
            elif avg_grade <= 14: level, color = "Advanced (High School/College)", "#ffc107"
            else: level, color = "Expert (Professional/Academic)", "#dc3545"

            st.markdown(f"""
            <div style="background-color: #1f2937; padding: 20px; border-radius: 10px; border-left: 5px solid {color}; text-align: center;">
                <h2 style="margin:0; color: {color} !important;">Overall Level: {level}</h2>
                <p style="margin:5px 0 0 0; color: #9ca3af;">Approximate Grade Level: {int(avg_grade)}</p>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("### 📈 Detailed Metrics")

            # 2. Visual Gauges
            c1, c2, c3 = st.columns(3)
            with c1:
                st.plotly_chart(create_gauge(score["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100, "#00ffcc"), use_container_width=True)
                with st.expander("ℹ️ About Flesch Ease"):
                    st.caption("0-100 Scale. Higher is easier. 60-70 is standard.")

            with c2:
                st.plotly_chart(create_gauge(score["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20, "#ff00ff"), use_container_width=True)
                with st.expander("ℹ️ About Kincaid Grade"):
                    st.caption("US Grade Level. 8.0 means 8th grader can understand.")

            with c3:
                st.plotly_chart(create_gauge(score["SMOG Index"], "SMOG Index", 0, 20, "#ffff00"), use_container_width=True)
                with st.expander("ℹ️ About SMOG"):
                    st.caption("Commonly used for medical writing. Based on polysyllables.")

            c4, c5 = st.columns(2)
            with c4:
                st.plotly_chart(create_gauge(score["Gunning Fog"], "Gunning Fog", 0, 20, "#00ccff"), use_container_width=True)
                with st.expander("ℹ️ About Gunning Fog"):
                    st.caption("Based on sentence length and complex words.")

            with c5:
                st.plotly_chart(create_gauge(score["Coleman-Liau"], "Coleman-Liau", 0, 20, "#ff9900"), use_container_width=True)
                with st.expander("ℹ️ About Coleman-Liau"):
                    st.caption("Based on characters instead of syllables. Good for automated analysis.")

            # 3. Text Stats
            st.markdown("### 📝 Text Statistics")
            s1, s2, s3, s4, s5 = st.columns(5)
            s1.metric("Sentences", analyzer.num_sentences)
            s2.metric("Words", analyzer.num_words)
            s3.metric("Syllables", analyzer.num_syllables)
            s4.metric("Complex Words", analyzer.complex_words)
            s5.metric("Characters", analyzer.char_count)

def _clear_stale_results(new_menu):
    """Clear previous results when switching between menu items"""
    if st.session_state.get('current_menu') != new_menu:
        # Clear summarization state
        for key in ['last_summary', 'last_summary_text', 'summarization_history']:
            if key in st.session_state:
                del st.session_state[key]
        # Clear paraphrasing state
        for key in ['last_para', 'last_para_text', 'paraphrasing_history']:
            if key in st.session_state:
                del st.session_state[key]
        st.session_state['current_menu'] = new_menu

def extract_text(file):
    try:
        if file.type == "application/pdf":
            reader = PyPDF2.PdfReader(file)
            return "".join([page.extract_text() + "\n" for page in reader.pages])
        else:
            return file.read().decode("utf-8")
    except Exception as e:
        st.error(f"Error reading file: {e}")
        return ""

def render_feedback_ui(email, original_text, generated_text, task_type):

    base_key = f"{task_type}_{hash(str(original_text)[:20])}"
    rating_key = f"r_{base_key}"
    comment_key = f"c_{base_key}"
    button_key = f"fbs_{base_key}"
    reset_key = f"reset_{base_key}"

    # ✅ STEP 1: Reset BEFORE widgets render
    if st.session_state.get(reset_key, False):
        st.session_state[rating_key] = None
        st.session_state[comment_key] = ""
        st.session_state[reset_key] = False  # clear flag

    with st.expander("📝 Provide Feedback"):
        col1, col2 = st.columns([1, 4])

        with col1:
            rating = st.radio(
                "Rating",
                [1, 2, 3, 4, 5],
                horizontal=True,
                key=rating_key
            )

        with col2:
            comments = st.text_input(
                "Comments (optional)",
                key=comment_key
            )

        if st.button("Submit Feedback", key=button_key):
            db.save_feedback(email, original_text, generated_text, task_type, rating, comments)

            st.success("Thank you for your feedback!")

            # ✅ STEP 2: Set reset flag
            st.session_state[reset_key] = True

            # 🔥 Force rerun
            st.rerun()

def summarizer_page():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_profile"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.title("📝 Multi-level Summarization")

    if 'summarization_history' not in st.session_state:
        st.session_state.summarization_history = []

    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Input Text")
        text_input = st.text_area("Enter text to summarize (min 50 chars):", height=200, key="summarization_text")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt", "pdf"], key="sum_upload")
        if uploaded_file:
            text_input = extract_text(uploaded_file)
            st.info(f"✅ File loaded ({len(text_input.split())} words)")

    with col2:
        st.subheader("Settings")
        summary_length = st.selectbox("Summary Length", ["Short", "Medium", "Long"])
        model_type = st.selectbox("Model", ["FLAN-T5", "BART", "Pegasus"])
        target_lang = st.selectbox("🌐 Output Language", SUPPORTED_LANGUAGES)

        if st.button("Generate Summary", type="primary", use_container_width=True):
            if len(text_input) < 50:
                st.error("Text is too short.")
            else:
                with st.spinner("Generating summary..."):
                    summary = engine.local_summarize(
                        text_input, summary_length, model_type,
                        st.session_state.summarization_models,
                        target_lang=target_lang
                    )

                    st.session_state.last_summary = summary
                    st.session_state.last_summary_text = text_input
                    st.session_state.last_summary_lang = target_lang

                    db.log_activity(
                        st.session_state['user'],
                        "Summarization",
                        f"Length: {summary_length}, Lang: {target_lang}, Input: {text_input[:50]}...",
                        summary,
                        model_type,
                        target_lang
                    )
                    st.session_state.summarization_history.append({
                        'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        'input': text_input[:100] + "..." if len(text_input) > 100 else text_input,
                        'summary': summary,
                        'length': summary_length,
                        'model': model_type,
                        'lang': target_lang
                    })

    if 'last_summary' in st.session_state:
        st.markdown("---")
        st.header("📋 Summary Results")

        if st.session_state.get('last_summary_lang', 'English') != 'English':
            st.info(f"🌐 Output translated to **{st.session_state.get('last_summary_lang', 'English')}**")

        c1, c2 = st.columns(2)
        with c1:
            st.subheader("📄 Original Text")
            st.info(st.session_state.last_summary_text)
            st.caption(f"**Word Count:** {len(st.session_state.last_summary_text.split())}")
        with c2:
            st.subheader("📝 Generated Summary")
            st.success(st.session_state.last_summary)
            st.caption(f"**Word Count:** {len(st.session_state.last_summary.split())}")
        user = st.session_state.get('user', 'Unknown User')
        render_feedback_ui(st.session_state['user'], st.session_state['last_summary_text'], st.session_state['last_summary'], "Summarization")

        with st.expander("📜 Summarization Session History"):
            if st.session_state.summarization_history:
                for item in reversed(st.session_state.summarization_history[-5:]):
                    lang_badge = f" 🌐 {item.get('lang', 'English')}" if item.get('lang', 'English') != 'English' else ""
                    st.write(f"**{item['timestamp']}** - {item['length']} ({item['model']}){lang_badge}")
                    st.info(f"Input: {item['input']}")
                    st.success(f"Summary: {item['summary']}")
                    st.caption(f"Words: {len(item['input'].split())} ➡️ {len(item['summary'].split())}")
                    st.markdown("---")

def paraphraser_page():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_profile"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.title("🔄 Advanced Paraphrasing Engine")

    if 'paraphrasing_history' not in st.session_state:
        st.session_state.paraphrasing_history = []

    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Input Text")
        text_input = st.text_area("Enter text to paraphrase (min 50 chars):", height=200, key="para_text")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt", "pdf"], key="para_upload")
        if uploaded_file:
            text_input = extract_text(uploaded_file)
            st.info(f"✅ File loaded ({len(text_input.split())} words)")

    with col2:
        st.subheader("Settings")
        complexity = st.selectbox("Complexity Level", ["Simple", "Neutral", "Advanced"])
        style = st.selectbox("Paraphrasing Style", ["Simplification", "Formalization", "Creative"])
        model_type = st.selectbox("Model", ["FLAN-T5", "BART"])
        target_lang = st.selectbox("🌐 Output Language", SUPPORTED_LANGUAGES, key="para_lang")

        if st.button("Generate Paraphrase", type="primary", use_container_width=True):
            if len(text_input) < 50:
                st.error("Text is too short.")
            else:
                with st.spinner("Generating paraphrase..."):
                    paraphrased = engine.paraphrase_with_model(
                        text_input, complexity, style, model_type,
                        st.session_state.paraphrase_models,
                        target_lang=target_lang
                    )

                    st.session_state.last_para = paraphrased
                    st.session_state.last_para_text = text_input
                    st.session_state.last_para_lang = target_lang

                    db.log_activity(
                        st.session_state['user'],
                        "Paraphrasing",
                        f"Complexity: {complexity}, Style: {style}, Input: {text_input[:50]}...",
                        paraphrased,
                        model_type,
                        target_lang
                    )
                    st.session_state.paraphrasing_history.append({
                        'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        'input': text_input[:100] + "..." if len(text_input) > 100 else text_input,
                        'paraphrase': paraphrased,
                        'complexity': complexity,
                        'style': style,
                        'model': model_type,
                        'lang': target_lang
                    })

    if 'last_para' in st.session_state:
        st.markdown("---")
        st.header("📋 Paraphrase Results")

        if st.session_state.get('last_para_lang', 'English') != 'English':
            st.info(f"🌐 Output translated to **{st.session_state.get('last_para_lang', 'English')}**")

        c1, c2 = st.columns(2)
        with c1:
            st.subheader("📄 Original Text")
            st.info(st.session_state.last_para_text)
            st.caption(f"**Word Count:** {len(st.session_state.last_para_text.split())}")
        with c2:
            st.subheader("🔄 Paraphrased Text")
            st.success(st.session_state.last_para)
            st.caption(f"**Word Count:** {len(st.session_state.last_para.split())}")
        user = st.session_state.get('user', 'Unknown User')
        render_feedback_ui(st.session_state['user'], st.session_state['last_para_text'], st.session_state['last_para'], "Paraphrasing")

        with st.expander("📜 Paraphrasing Session History"):
            if st.session_state.paraphrasing_history:
                for item in reversed(st.session_state.paraphrasing_history[-5:]):
                    lang_badge = f" 🌐 {item.get('lang', 'English')}" if item.get('lang', 'English') != 'English' else ""
                    st.write(f"**{item['timestamp']}** - {item['complexity']} ({item['style']}) - {item['model']}{lang_badge}")
                    st.info(f"Input: {item['input']}")
                    st.success(f"Paraphrase: {item['paraphrase']}")
                    st.caption(f"Words: {len(item['input'].split())} ➡️ {len(item['paraphrase'].split())}")
                    st.markdown("---")
import streamlit as st
import pandas as pd
import plotly.express as px

def history_page():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_profile"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.title("📜 Activity History Dashboard")
    # Get user activity
    activities = db.get_user_activity(st.session_state['user'])

    if not activities:
        st.info("No activity history yet. Start using the features!")
        return

    import plotly.express as px

    # Create dataframe
    df = pd.DataFrame(
        activities,
        columns=["Activity Type", "Details", "Output", "Model Used", "Timestamp"]
    )

    df = df[df["Activity Type"] != "Login"]

    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    # ---------------- FILTERS ----------------
    st.subheader("🎛 Filters")

    col1, col2 = st.columns(2)

    with col1:
        activity_filter = st.selectbox(
            "Filter by Activity",
            ["All"] + list(df["Activity Type"].unique())
        )

    with col2:
        model_filter = st.selectbox(
            "Filter by Model",
            ["All"] + list(df["Model Used"].unique())
        )

    if activity_filter != "All":
        df = df[df["Activity Type"] == activity_filter]

    if model_filter != "All":
        df = df[df["Model Used"] == model_filter]

    # ---------------- USER PROFILE ----------------
    st.markdown("---")
    st.subheader("👤 User Profile")

    username = db.get_username(st.session_state['user'])

    total_actions = len(df)
    unique_models = df["Model Used"].nunique()

    col1, col2, col3 = st.columns(3)

    col1.metric("Username", username)
    col2.metric("Total Activities", total_actions)
    col3.metric("Models Used", unique_models)
    st.markdown("---")

    # ---------------- DATA TABLE ----------------
    st.subheader("📋 Activity Records")
    if not activities:
        st.info("No activity history yet. Start using the features!")
        return

    df = pd.DataFrame(
        activities,
        columns=["Activity Type", "Details", "Output", "Model Used", "Timestamp"]
    )

    # ---------- DOWNLOAD BUTTON (ONLY ONCE) ----------
    csv = df.to_csv(index=False).encode("utf-8")

    st.download_button(
        label="⬇ Download History (CSV)",
        data=csv,
        file_name="history.csv",
        mime="text/csv",
        use_container_width=True,
        key="download_history_csv"
    )

    st.markdown("---")

    # ---------- ACTIVITY CARDS ----------
    for i, row in df.iterrows():

        icon = "🧠" if row["Activity Type"] == "Summarization" else "🔄"

        details_preview = row["Details"][:120] + "..." if len(row["Details"]) > 120 else row["Details"]

        st.markdown(f"""
        <div style="
            background: rgba(255,255,255,0.05);
            padding:20px;
            border-radius:12px;
            border:1px solid #00F5FF;
            margin-bottom:15px;
            box-shadow:0 0 10px rgba(0,245,255,0.3);
        ">
        <h4>{icon} {row["Activity Type"]}</h4>

        <p><b>🤖 Model:</b> {row["Model Used"]}</p>

        <p><b>📅 Date:</b> {row["Timestamp"]}</p>

        <p><b>📄 Details:</b> {details_preview}</p>

        </div>
        """, unsafe_allow_html=True)

        with st.expander("🔍 View Full Output"):
            st.markdown("**📥 Input Text**")
            st.info(row["Details"])

            st.markdown("**📤 AI Output**")
            st.success(row["Output"])

    # ---------------- ANALYTICS ----------------
    st.markdown("---")
    st.subheader("📊 Usage Analytics")

    col1, col2 = st.columns(2)

    # Activity Distribution
    activity_counts = df["Activity Type"].value_counts().reset_index()
    activity_counts.columns = ["Activity", "Count"]

    fig1 = px.bar(
        activity_counts,
        x="Activity",
        y="Count",
        title="📊 Activity Distribution",
        text="Count"
    )

    col1.plotly_chart(fig1, use_container_width=True)

    # Model Usage
    model_counts = df["Model Used"].value_counts().reset_index()
    model_counts.columns = ["Model", "Usage"]

    fig2 = px.bar(
        model_counts,
        x="Model",
        y="Usage",
        title="🔥 Top Used Models",
        text="Usage"
    )

    col2.plotly_chart(fig2, use_container_width=True)


    # -------- Activity Timeline --------
    st.subheader("📅 Activity Timeline")

    timeline = df.copy()

    # Convert timestamp to datetime
    timeline["Timestamp"] = pd.to_datetime(timeline["Timestamp"], errors="coerce")

    timeline["Date"] = timeline["Timestamp"].dt.date

    timeline_counts = timeline.groupby("Date").size().reset_index(name="Actions")

    fig = px.line(
        timeline_counts,
        x="Date",
        y="Actions",
        markers=True,
        title="Daily Activity Trend"
    )

    st.plotly_chart(fig, use_container_width=True)

def _simulate_training_metrics(model_arch, epochs, learning_rate, batch_size, dropout_rate, quantization):
    """Generate dynamic training metrics based on user config instead of hardcoded values"""
    random.seed(hash(f"{model_arch}{epochs}{learning_rate}{batch_size}{dropout_rate}{quantization}"))

    lr_val = float(learning_rate)

    # Base loss depends on model architecture
    base_loss = {"T5-Small": 0.55, "BART-Base": 0.48, "FLAN-T5": 0.42}.get(model_arch, 0.50)

    # More epochs = lower loss (with diminishing returns)
    epoch_factor = 1.0 - (min(epochs, 10) * 0.06)

    # Learning rate effect
    lr_factor = 1.0 - (lr_val * 8000)

    # Dropout regularization
    dropout_bonus = dropout_rate * 0.08

    # Quantization impact
    quant_penalty = {"FP16 (None)": 0.0, "8-bit": 0.02, "4-bit": 0.05}.get(quantization, 0.0)

    final_loss = round(max(0.15, base_loss * epoch_factor * lr_factor + dropout_bonus + quant_penalty + random.uniform(-0.03, 0.03)), 2)
    delta_loss = round(random.uniform(-0.08, -0.15), 2)

    accuracy = round(min(95, 65 + (epochs * 2.5) + (1 - final_loss) * 20 + random.uniform(-2, 3)), 1)
    delta_acc = f"+{round(random.uniform(1, 6), 1)}%"

    rouge_l = round(random.uniform(1.5, 4.0) + epochs * 0.15, 1)
    delta_rouge = f"+{round(random.uniform(0.3, 1.2), 1)}"

    bleu = round(0.25 + (epochs * 0.02) + (1 - final_loss) * 0.15 + random.uniform(-0.03, 0.03), 2)
    delta_bleu = f"+{round(random.uniform(0.02, 0.08), 2)}"

    # Generate loss curve
    loss_curve = []
    curr_loss = base_loss + 1.0
    for i in range(epochs):
        decay = 0.6 + random.uniform(-0.05, 0.05)
        curr_loss = curr_loss * decay + random.uniform(-0.02, 0.02)
        loss_curve.append(round(max(final_loss, curr_loss), 3))
    loss_curve[-1] = final_loss

    return {
        "final_loss": final_loss, "delta_loss": str(delta_loss),
        "accuracy": f"{accuracy}%", "delta_acc": delta_acc,
        "rouge_l": f"+{rouge_l}", "delta_rouge": delta_rouge,
        "bleu": str(bleu), "delta_bleu": delta_bleu,
        "loss_curve": loss_curve, "epochs_x": list(range(1, epochs + 1))
    }


def augmentation_page():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_profile"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.title("🗃️ Dataset Augmentation & Custom Model Tuning")
    st.info("🚀 Manage datasets, visualize distributions, and fine-tune custom models.")

    # Data Explorer Tab & Tuning Tab
    tab_explore, tab_tune, tab_studio = st.tabs(["📊 Dataset Explorer", "🛠️ Model Tuning", "🧪 Augmentation Studio"])

    with tab_explore:
        st.subheader("Data Inspector & Cleaner")
        datasets = {
            "CNN/DailyMail": {"samples": 311029, "type": "News Summarization", "avg_words": 781},
            "XSum": {"samples": 226711, "type": "Extreme Summarization", "avg_words": 431},
            "PAWS": {"samples": 108461, "type": "Paraphrase", "avg_words": 21}
        }
        selected_dataset = st.selectbox("Select Active Dataset", list(datasets.keys()))

        c1, c2, c3 = st.columns(3)
        with c1: st.metric("Total Samples", f"{datasets[selected_dataset]['samples']:,}")
        with c2: st.metric("Task Type", datasets[selected_dataset]["type"])
        with c3: st.metric("Avg Document Length", f"{datasets[selected_dataset]['avg_words']} words")

        st.markdown("### 🧹 Interactive Data Cleaning")
        clean_col1, clean_col2 = st.columns(2)
        with clean_col1:
            min_length = st.slider("Filter Minimum Words", 5, 100, 10)
        with clean_col2:
            max_length = st.slider("Filter Maximum Words", 100, 2000, 1000)

        raw_samples = datasets[selected_dataset]['samples']
        filtered_samples = int(raw_samples * (0.9 - (min_length/1000) - (1000-max_length)/2000))
        st.success(f"✅ Filter applied! Current Cleaned Dataset Size: **{filtered_samples:,} valid pairs** prepared for training.")

        st.markdown("### 👁️ Dataset Preview View")
        mock_data = {
            "ID": [f"{selected_dataset[:3]}-001", f"{selected_dataset[:3]}-002", f"{selected_dataset[:3]}-003", f"{selected_dataset[:3]}-004"],
            "Original Text": [f"Sample text sequence {i} from {selected_dataset} containing unstructured content." for i in range(1, 5)],
            "Target Summary/Paraphrase": [f"Cleaned target pair {i} optimized for AI." for i in range(1, 5)],
            "Word Count": [140, 432, 21, 89],
            "Complexity Score": [8.4, 12.1, 4.2, 7.9]
        }

        df_preview = pd.DataFrame(mock_data)

        for _, row in df_preview.iterrows():

            st.markdown(f"""
            <div style="
                background: rgba(255,255,255,0.04);
                padding:18px;
                border-radius:12px;
                border:1px solid #00F5FF;
                margin-bottom:15px;
                box-shadow:0 0 8px rgba(0,245,255,0.25);
            ">

            <h4 style="color:#00F5FF;">🆔 {row['ID']}</h4>

            <p><b>📄 Original Document</b><br>
            {row['Original Text']}</p>

            <p><b>🤖 Target Output</b><br>
            {row['Target Summary/Paraphrase']}</p>

            <p>
            📊 <b>Words:</b> {row['Word Count']} &nbsp;&nbsp;
            ⚡ <b>Complexity:</b> {row['Complexity Score']}
            </p>

            </div>
            """, unsafe_allow_html=True)

    with tab_tune:
        st.subheader("🛠️ Model Configuration Matrix")
        c1, c2, c3 = st.columns(3)
        with c1:
            model_arch = st.selectbox("Model Architecture", ["T5-Small", "BART-Base", "FLAN-T5"])
            epochs = st.slider("Training Epochs", 1, 10, 3)
        with c2:
            quantization = st.selectbox("Quantization (BitsAndBytes)", ["FP16 (None)", "8-bit", "4-bit"])
            batch_size = st.slider("Batch Size", 8, 32, 16)
        with c3:
            learning_rate = st.selectbox("Learning Rate", ["1e-5", "2e-5", "3e-5"])
            dropout_rate = st.slider("Dropout", 0.0, 0.5, 0.1)

        if st.button("🚀 Execute Distributed Training", type="primary", use_container_width=True):
            with st.spinner(f"Allocating GPU resources & Tuning {model_arch} (Q: {quantization})..."):
                progress_bar = st.progress(0)
                for i in range(100):
                    time.sleep(0.01)
                    progress_bar.progress(i + 1)

                st.success(f"✅ Custom Model {model_arch} compiled & saved to /models/custom_{selected_dataset.replace('/','').lower()}/")

                # Dynamic metrics based on user configuration
                metrics = _simulate_training_metrics(model_arch, epochs, learning_rate, batch_size, dropout_rate, quantization)

                st.markdown("### 📊 Validation Report")

                m1, m2, m3, m4 = st.columns(4)
                with m1: st.metric("Final Epoch Loss", metrics["final_loss"], metrics["delta_loss"])
                with m2: st.metric("Train Accuracy", metrics["accuracy"], metrics["delta_acc"])
                with m3: st.metric("ROUGE-L Delta", metrics["rouge_l"], metrics["delta_rouge"])
                with m4: st.metric("BLEU Score", metrics["bleu"], metrics["delta_bleu"])

                # Plotly Chart showing dynamic loss curve
                fig = go.Figure(data=go.Scatter(
                    x=metrics["epochs_x"], y=metrics["loss_curve"],
                    mode='lines+markers',
                    line=dict(color='#00ffcc', width=3),
                    marker=dict(size=8, color='#00ffcc')
                ))
                fig.update_layout(
                    title=f"Training Loss Curve — {model_arch} ({quantization})",
                    xaxis_title="Epoch", yaxis_title="Cross-Entropy Loss",
                    template="plotly_dark", height=300,
                    margin=dict(l=0,r=0,t=40,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)

                user = st.session_state.get('user', 'Unknown User')

                db.log_activity(
                    user,
                    "Model Training",
                    f"Trained {model_arch} on {selected_dataset}",
                    f"Loss: {metrics.get('final_loss', 'N/A')}, Config: {quantization}",
                    model_arch   # ✅ ADD THIS
                )

    with tab_studio:
        st.subheader("🧪 Live Dataset Pair Generator (Batch Processing)")
        st.info("Input multiple paragraphs below (separated by blank lines). The AI will process each into a high-quality dataset pair ready for export.")

        aug_input = st.text_area(
            "Original Text (Paste multiple paragraphs here):",
            height=200,
            key="aug_text_input",
            value="The quick brown fox jumps over the lazy dog.\n\nArtificial Intelligence is rapidly evolving in the modern era."
        )

        c_aug1, c_aug2 = st.columns(2)

        with c_aug1:
            aug_type = st.selectbox(
                "Transformation Type",
                ["Paraphrasing", "Summarization"],
                key="aug_type"
            )

        with c_aug2:
            if aug_type == "Summarization":
                aug_setting = st.selectbox("Length", ["Short", "Medium", "Long"], key="aug_setting")
            else:
                aug_setting = st.selectbox("Complexity", ["Advanced", "Simple", "Neutral"], key="aug_setting")

        if st.button("Generate Dataset 🚀", type="secondary", use_container_width=True):

            paragraphs = [p.strip() for p in aug_input.split('\n\n') if len(p.strip()) > 10]

            if not paragraphs:
                st.error("Please enter at least one valid paragraph.")
            else:
                results = []
                progress_text = "Processing your dataset..."
                my_bar = st.progress(0, text=progress_text)

                with st.spinner(f"Batch processing {len(paragraphs)} pairs..."):

                    for idx, para in enumerate(paragraphs):

                        if aug_type == "Summarization":
                            res = engine.local_summarize(
                                para, aug_setting, "BART",
                                st.session_state.summarization_models
                            )
                            icon = "🧠"
                        else:
                            res = engine.paraphrase_with_model(
                                para, aug_setting, "Creative", "FLAN-T5",
                                st.session_state.paraphrase_models
                            )
                            icon = "🔄"

                        orig_wc = len(para.split())
                        target_wc = len(res.split())
                        delta = target_wc - orig_wc

                        results.append({
                            "#": idx + 1,
                            "Original Text": para[:200] + ("..." if len(para) > 200 else ""),
                            "Target Text": res[:200] + ("..." if len(res) > 200 else ""),
                            "Orig Words": orig_wc,
                            "Target Words": target_wc,
                            "Delta": f"{delta:+d}",
                            "Full Original": para,
                            "Full Target": res,
                            "Icon": icon
                        })

                        my_bar.progress(
                            (idx + 1) / len(paragraphs),
                            text=f"Processed: {idx+1}/{len(paragraphs)}"
                        )

                st.success(f"✅ Successfully generated {len(results)} pairs!")

                # ================= NEW UI (CARDS LIKE HISTORY) =================
                st.markdown("---")
                st.subheader("📋 Generated Dataset")

                for row in results:

                    st.markdown(f"""
                    <div style="
                        background: rgba(255,255,255,0.05);
                        padding:20px;
                        border-radius:12px;
                        border:1px solid #00F5FF;
                        margin-bottom:15px;
                        box-shadow:0 0 10px rgba(0,245,255,0.3);
                    ">
                    <h4>{row["Icon"]} Pair #{row["#"]}</h4>

                    <p><b>📊 Original Words:</b> {row["Orig Words"]}</p>
                    <p><b>📊 Generated Words:</b> {row["Target Words"]}</p>
                    <p><b>📈 Change:</b> {row["Delta"]}</p>

                    <p><b>📄 Original Text:</b> {row["Original Text"]}</p>
                    <p><b>✨ Generated Text:</b> {row["Target Text"]}</p>

                    </div>
                    """, unsafe_allow_html=True)

                    with st.expander(f"🔍 View Full Pair #{row['#']}"):

                        st.markdown("**📥 Full Original Text**")
                        st.info(row["Full Original"])

                        st.markdown("**📤 Full Generated Text**")
                        st.success(row["Full Target"])

                # ================= CSV DOWNLOAD =================
                full_results = [
                    {"Original Text": r["Full Original"], "Target Text": r["Full Target"]}
                    for r in results
                ]

                csv = pd.DataFrame(full_results).to_csv(index=False).encode('utf-8')

                st.download_button(
                    label="📥 Download Dataset (CSV)",
                    data=csv,
                    file_name='augmented_dataset.csv',
                    mime='text/csv',
                    use_container_width=True
                )

                # ================= ACTIVITY LOG =================
                user = st.session_state.get('user', 'Unknown User')

                db.log_activity(
                    user,
                    "Batch Augmentation",
                    f"Type: {aug_type}, Setting: {aug_setting}, Samples: {len(results)}",
                    str(full_results),
                    "Augmentation Engine",
                    "Multi"
                )

    st.markdown("---")

    user = st.session_state.get('user', 'Unknown User')

    render_feedback_ui(
        user,
        "Dataset Augmentation Module",
        "N/A",
        "Dataset Augmentation"
    )

def _auth_header():
    st.markdown("""
    <div style="
        text-align:center;
        margin-bottom:10px;   /* ⬅ reduced */
    ">
        <h1 style="
            color:#00F5FF;
            font-size:40px;
            margin-bottom:2px;   /* ⬅ tighter */
        ">
            🔐 TextMorph
        </h1>
        <p style="
            color:#6B8BA4;
            font-size:15px;
            margin:0;           /* ⬅ remove extra gap */
        ">
            Secure Authentication Portal
        </p>
    </div>
    """, unsafe_allow_html=True)
def admin_home_page():

    conn = sqlite3.connect(db.DB_NAME)
    try:
        conn.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'")
        conn.commit()
    except:
        pass

    st.markdown(f"""
    <style>
    [data-testid="block-container"] {{

        padding-top: 1rem;
    }}

    .stats-row {{
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        gap: 16px;
        width: 100%;
        margin-top: 4px;
    }}

    @media (max-width: 680px) {{
        .stats-row {{ grid-template-columns: repeat(2, 1fr); }}
    }}

    .stat-box {{
        background: rgba(13,24,46,0.95);
        border: 1px solid rgba(0,200,255,0.28);
        border-radius: 14px;
        padding: 18px 14px;
        text-align: center;
    }}

    .stat-num {{
        font-family:'Syne',sans-serif;
        font-size:1.8rem;
        font-weight:800;
        color:#00C8FF;
    }}

    .stat-label {{
        font-size:.78rem;
        color:#6B8BA4;
        margin-top:4px;
    }}
    </style>
    """, unsafe_allow_html=True)

    # Header
    st.markdown("""
    <div style="display:flex;align-items:center;gap:14px;margin-bottom:28px">
      <span style="font-size:2.2rem">🛠️</span>
      <div>
        <p style="font-weight:800;font-size:1.7rem;color:#00C8FF;margin:0;">
        Admin Dashboard</p>
        <p style="font-size:.85rem;color:#6B8BA4;">
        Manage users, analytics, and system</p>
      </div>
    </div>
    """, unsafe_allow_html=True)

    # Tools
    ADMIN_TOOLS = [
        ("👥", "Users", "delete users, manage roles, and control user access permissions"),
        ("📊", "Analytics", "View system insights, usage trends, performance metrics, and user statistics"),
        ("📜", "Activity", "Track all user actions, login history, system logs, and recent activities"),
        ("❌", "Remove Admin", "Revoke admin privileges and convert administrators back to normal users securely"),
        ("💬", "Feedback", "Review user feedback, ratings, suggestions, and improve system experience"),
        ("🔒", "Locked", "Manage locked accounts, unlock users, and monitor suspicious login attempts"),
        ("⬇️", "Download", "Export system data, reports, and user records in CSV or Excel format"),
    ]

    st.markdown("""
    <style>
    div[data-testid="stButton"] > button {
        background: linear-gradient(145deg, #0F1B2D, #0C1828) !important;
        border: 1px solid rgba(0,200,255,0.28) !important;
        border-radius: 18px !important;
        padding: 32px 20px 28px !important;
        min-height: 200px !important;
        display: flex !important;
        flex-direction: column !important;
        align-items: center !important;
        justify-content: center !important;
        gap: 10px !important;
        color: #E8F4FD !important;
        font-size: 0.84rem !important;
        font-family: 'DM Sans', sans-serif !important;
        white-space: normal !important;
        line-height: 1.5 !important;
        text-align: center !important;
        box-shadow: none !important;
        transition: all 0.25s ease !important;
    }

    div[data-testid="stHorizontalBlock"] div[data-testid="stButton"] > button:hover {
        border-color: #00C8FF !important;
        transform: translateY(-5px) !important;
        box-shadow: 0 10px 32px rgba(0,200,255,0.28) !important;
        background: linear-gradient(145deg, #112240, #0F1B2D) !important;
    }
    </style>
    """, unsafe_allow_html=True)

    # Grid
    cols = st.columns(3, gap="large")

    for i, (icon, title, desc) in enumerate(ADMIN_TOOLS):
        with cols[i % 3]:
            if st.button(f"{icon}\n\n**{title}**\n\n{desc}", key=f"admin_{title}"):
                st.session_state["_nav_to"] = title
                st.rerun()

    st.markdown("<hr>", unsafe_allow_html=True)

    # ================= FIXED STATS =================

    # Count admins
    total_admins = conn.execute("""
        SELECT COUNT(*)
        FROM users u
        LEFT JOIN user_roles r ON u.email = r.email
        WHERE r.role = 'admin'
        OR u.email = 'admin@textmorph.com'
    """).fetchone()[0]

    # ✅ Count ONLY normal users (exclude admins completely)
    total_users = conn.execute("""
        SELECT COUNT(*)
        FROM users u
        LEFT JOIN user_roles r ON u.email = r.email
        WHERE COALESCE(r.role, 'user') != 'admin'
        AND u.email != 'admin@textmorph.com'
    """).fetchone()[0]

    # Activities
    activities = db.get_all_activity() or []
    total_acts = len(activities)

    # ================= DISPLAY =================

    st.markdown(f"""
    <div class="stats-row">
      <div class="stat-box"><div class="stat-num">{total_users}</div><div class="stat-label">Users</div></div>
      <div class="stat-box"><div class="stat-num">{total_admins}</div><div class="stat-label">Admins</div></div>
      <div class="stat-box"><div class="stat-num">{total_acts}</div><div class="stat-label">Activities</div></div>
      <div class="stat-box"><div class="stat-num">Live</div><div class="stat-label">Status</div></div>
    </div>
    """, unsafe_allow_html=True)

    conn.close()
def admin_page():
    if selected == "Admin Dashboard":
        admin_dashboard()

    elif selected == "User Management":
        user_management()

    elif selected == "Activity Tracking":
        activity_tracking()

    elif selected == "Feedback":
        feedback_section()

    elif selected == "Locked Accounts":
        locked_accounts_section()
    elif selected == "Export Data":
        export_data()
def remove_admin():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.subheader("🛑 Remove Admin Access")
    conn = sqlite3.connect(db.DB_NAME)

    admins = pd.read_sql_query("""
    SELECT email
    FROM user_roles
    WHERE role='admin'
    """, conn)

    if admins.empty:
        st.info("No admin users found.")
        return

    selected_admin = st.selectbox("Select Admin to Remove", admins["email"])

    if st.button("Remove Admin Privilege"):

        conn.execute(
            "DELETE FROM user_roles WHERE email=?",
            (selected_admin,)
        )

        conn.commit()

        st.success("Admin privileges removed")
        st.rerun()
def export_data():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.subheader("⬇ Export System Data")

    conn = sqlite3.connect(db.DB_NAME)

    users = pd.read_sql_query("SELECT * FROM users", conn)

    activities = pd.read_sql_query(
        "SELECT * FROM activity_history",
        conn
    )

    feedback = pd.read_sql_query(
        "SELECT * FROM feedback",
        conn
    )

    deleted_users = pd.read_sql_query(
        "SELECT * FROM deleted_users",
        conn
    )

    conn.close()

    # Combine everything
    export_df = pd.concat([
        users.assign(type="active_user"),
        deleted_users.assign(type="deleted_user"),
        activities.assign(type="activity"),
        feedback.assign(type="feedback")
    ], ignore_index=True)

    csv = export_df.to_csv(index=False).encode()

    st.download_button(
        "📥 Download Full System Report",
        csv,
        "system_report.csv",
        "text/csv"
    )
def admin_dashboard():

    if "_nav_to" not in st.session_state:
        st.session_state["_nav_to"] = "Home"

    page = st.session_state["_nav_to"]

    # 🔹 HOME
    if page == "Home":
        admin_home_page()

    # 🔹 USERS
    elif page == "Users":
        user_management()

    # 🔹 ANALYTICS
    elif page == "Analytics":
        analytics_dashboard()

    # 🔹 ACTIVITY
    elif page == "Activity":
        activity_tracking()

    # 🔹 REMOVE ADMIN
    elif page == "Remove Admin":
        remove_admin()

    # 🔹 FEEDBACK
    elif page == "Feedback":
        feedback_section()

    # 🔹 LOCKED
    elif page == "Locked":
        locked_accounts_section()

    # 🔹 DOWNLOAD
    elif page == "Download":
        export_data()

def user_management():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.subheader("👥 User Management")
    conn = sqlite3.connect(db.DB_NAME)

    users = pd.read_sql_query("""
    SELECT u.email
    FROM users u
    LEFT JOIN user_roles r
    ON u.email = r.email
    WHERE COALESCE(r.role,'user') != 'admin'
    AND u.email != 'admin@textmorph.com'
    """, conn)



    if users.empty:
        st.info("No users available.")
        return

    selected_user = st.selectbox("Select User", users["email"])

    col1, col2 = st.columns(2)

    with col1:
        if st.button("Promote to Admin", use_container_width=True):

            conn.execute("""
            INSERT OR REPLACE INTO user_roles(email, role)
            VALUES (?, 'admin')
            """, (selected_user,))

            conn.commit()

            st.success("User promoted to Admin")
            st.rerun()

    with col2:
        if st.button("Delete User", use_container_width=True):

            conn.execute(
                "DELETE FROM users WHERE email=?",
                (selected_user,)
            )

            conn.commit()

            st.error("User Deleted")
            st.rerun()

    conn.close()
def locked_accounts_section():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.subheader("🔒 Locked Accounts")
    locked_users = db.get_locked_accounts()

    if not locked_users:
        st.success("No locked accounts.")
    else:

        df = pd.DataFrame(
            locked_users,
            columns=["Email", "Locked At"]
        )

        st.dataframe(df, use_container_width=True)

        for email, locked_at in locked_users:

            col1, col2 = st.columns([3,1])

            col1.write(email)

            if col2.button("Unlock", key=f"unlock_{email}"):

                db.unlock_account(email)

                st.success(f"{email} unlocked")
                st.rerun()

    st.markdown("---")

    st.subheader("⚙️ Manual Account Control")

    conn = sqlite3.connect(db.DB_NAME)

    emails_df = pd.read_sql_query("""
    SELECT u.email
    FROM users u
    LEFT JOIN user_roles r
    ON u.email = r.email
    WHERE COALESCE(r.role,'user') != 'admin'
    AND u.email != 'admin@textmorph.com'
    """, conn)

    conn.close()

    emails = emails_df["email"].tolist()

    if not emails:
        st.info("No users available.")
        return

    selected_user = st.selectbox("Select User", emails)

    col1, col2 = st.columns(2)

    if col1.button("🔒 Lock Account"):

        db.lock_account(selected_user)

        st.warning(f"{selected_user} account locked")
        st.rerun()

    if col2.button("🔓 Unlock Account"):

        db.unlock_account(selected_user)

        st.success(f"{selected_user} account unlocked")
        st.rerun()

    st.markdown("---")
def analytics_dashboard():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.title("📊 System Analytics")

    conn = sqlite3.connect(db.DB_NAME)

    activity = pd.read_sql_query(
        """
        SELECT email,
               activity_type,
               model_used,
               created_at
        FROM activity_history
        """,
        conn
    )

    if activity.empty:
        st.info("No activity data available yet.")
        return

    # ---------- Feature Popularity ----------

    st.subheader("⭐ Feature Popularity")

    feature_counts = activity["activity_type"].value_counts()

    fig1 = go.Figure(
        data=[
            go.Bar(
                x=feature_counts.index,
                y=feature_counts.values
            )
        ]
    )

    fig1.update_layout(
        xaxis_title="Feature",
        yaxis_title="Usage Count"
    )

    st.plotly_chart(fig1, use_container_width=True)

    # ---------- Model Usage ----------

    st.subheader("🤖 Model Usage")

    model_counts = activity["model_used"].dropna().value_counts()

    if not model_counts.empty:

        fig2 = go.Figure(
            data=[
                go.Pie(
                    labels=model_counts.index,
                    values=model_counts.values
                )
            ]
        )

        st.plotly_chart(fig2, use_container_width=True)

    else:
        st.info("No model usage data available.")

    # ---------- Daily Activity ----------

    st.subheader("📅 Daily Activity Trend")

    activity["created_at"] = pd.to_datetime(activity["created_at"])

    activity["date"] = activity["created_at"].dt.date

    daily_activity = activity.groupby("date").size()

    fig3 = go.Figure(
        data=[
            go.Scatter(
                x=daily_activity.index,
                y=daily_activity.values,
                mode="lines+markers"
            )
        ]
    )

    fig3.update_layout(
        xaxis_title="Date",
        yaxis_title="Activities"
    )

    st.plotly_chart(fig3, use_container_width=True)

def activity_tracking():
    import streamlit as st
    import pandas as pd
    import db

    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()

    st.subheader("📊 Activity Tracking")

    activities = db.get_all_activity()

    if not activities:
        st.info("No activity recorded yet.")
        return

    df = pd.DataFrame(activities, columns=[
        "Email",
        "Action",
        "Details",
        "Language",
        "Model Name",
        "Timestamp",
        "Output"
    ])

    df["Output Preview"] = df["Output"].apply(
        lambda x: x[:80] + "..." if isinstance(x, str) and len(x) > 80 else x
    )

    # ---------------- FILTERS ----------------
    col1, col2, col3 = st.columns(3)

    with col1:
        user_filter = st.selectbox(
            "Filter by User",
            ["All"] + sorted(df["Email"].unique())
        )

    with col2:
        action_filter = st.selectbox(
            "Filter by Action",
            ["All"] + sorted(df["Action"].unique())
        )

    with col3:
        model_filter = st.selectbox(
            "Filter by Model",
            ["All"] + sorted(df["Model Name"].unique())
        )

    filtered = df.copy()

    if user_filter != "All":
        filtered = filtered[filtered["Email"] == user_filter]

    if action_filter != "All":
        filtered = filtered[filtered["Action"] == action_filter]

    if model_filter != "All":
        filtered = filtered[filtered["Model Name"] == model_filter]

    # ---------------- METRICS ----------------
    col1, col2, col3 = st.columns(3)

    col1.metric("Total Activities", len(df))
    col2.metric("Active Users", df["Email"].nunique())
    col3.metric("Actions Logged", df["Action"].nunique())

    # ================= REPLACED TABLE =================
    st.markdown("---")
    st.subheader("📋 Activity Records")

    import re

    for i, row in filtered.iterrows():

        icon = "🧠" if row["Action"] == "Summarization" else \
              "🔄" if row["Action"] == "Paraphrasing" else \
              "⚙️"

        details_text = str(row["Details"])

        # ✅ Better Language Extraction
        language_display = row["Language"] if row["Language"] else "English"

        details_preview = details_text[:120] + "..." if len(details_text) > 120 else details_text

        st.markdown(f"""
        <div style="
            background: rgba(255,255,255,0.05);
            padding:20px;
            border-radius:12px;
            border:1px solid #00F5FF;
            margin-bottom:15px;
            box-shadow:0 0 10px rgba(0,245,255,0.3);
        ">
        <h4>{icon} {row["Action"]}</h4>

        <p><b>👤 User:</b> {row["Email"]}</p>

        <p><b>🌐 Language:</b> {language_display}</p>

        <p><b>🤖 Model:</b> {row["Model Name"]}</p>

        <p><b>📅 Date:</b> {row["Timestamp"]}</p>

        <p><b>📄 Details:</b> {details_preview}</p>

        </div>
        """, unsafe_allow_html=True)

        with st.expander("🔍 View Full Output"):
            st.markdown("**📥 Full Details**")
            st.info(details_text)

            st.markdown("**📤 Output**")
            st.success(row["Output"])   # ✅ FULL output now



def feedback_section():
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_unique_name"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()
    st.subheader("💬 User Feedback")

    conn = sqlite3.connect(db.DB_NAME)

    feedback = pd.read_sql_query(
        """
        SELECT email, task_type, rating, comments, created_at
        FROM feedback
        ORDER BY created_at DESC
        """,
        conn
    )
    if feedback.empty:
        st.info("No feedback available yet.")
        return

    # Combine comments for wordcloud
    text = " ".join(feedback["comments"].dropna())

    if text.strip():  # only generate if there is text
        wordcloud = WordCloud(
            width=800,
            height=400,
            background_color="black"
        ).generate(text)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(wordcloud, interpolation='bilinear')
        ax.axis("off")
        st.pyplot(fig)
    else:
        st.warning("No comments available to generate a word cloud.")

    for _, row in feedback.iterrows():

        st.markdown(f"""
        <div style="
            border:1px solid #00e5ff;
            border-radius:12px;
            padding:18px;
            margin-bottom:15px;
            background:rgba(0,40,60,0.6);
        ">

        <h3 style="color:#00e5ff;">🧠 {row['task_type']}</h3>

        <p>👤 <b>User:</b> {row['email']}</p>

        <p>⭐ <b>Rating:</b> {"⭐"*int(row['rating'])}</p>

        <p>💬 <b>Comment:</b> {row['comments'] if row['comments'] else "No comment"}</p>

        <p>📅 <b>Date:</b> {row['created_at']}</p>

        </div>
        """, unsafe_allow_html=True)
# ---------------- SIGNUP ----------------
def signup():

    st.markdown('<div class="neon-card">', unsafe_allow_html=True)

    st.markdown("""
    <h2 style="text-align:center;">🚀 Create Your Account</h2>
    """, unsafe_allow_html=True)

    questions = [
        "What is your pet name?",
        "What is your mother’s maiden name?",
        "What is your favorite teacher?",
        "What was your first school name?",
        "What is your favorite food?"
    ]

    _, col, _ = st.columns([1, 2, 1], gap="medium")

    with col:
        with st.form("signup_form"):

            c1, c2 = st.columns(2)

            with c1:
                username = st.text_input("👤 Username", placeholder="johndoe")

            with c2:
                email = st.text_input("📧 Email", placeholder="you@example.com")

            password = st.text_input("🔒 Password", type="password")
            confirm = st.text_input("🔒 Confirm Password", type="password")

            question = st.selectbox("❓ Security Question", questions)
            answer = st.text_input("✏️ Security Answer")

            submit = st.form_submit_button("✨ Create Account", use_container_width=True)

            if password:
                score = password_strength(password)
                st.text(f"Password strength: {score}/5")

            if submit:
                username = username.strip()
                email = email.strip()
                password = password.strip()
                confirm = confirm.strip()
                answer = answer.strip()

                if not username:
                    st.error("Username cannot be empty")
                elif not email:
                    st.error("Email cannot be empty")
                elif not valid_email(email):
                    st.error("Invalid Email format")
                elif not valid_password(password):
                    st.error("Weak password")
                elif password != confirm:
                    st.error("Passwords do not match")
                elif not answer:
                    st.error("Security Answer required")

                else:
                    existing_user = db.check_user_exists(email)

                    if existing_user:
                        st.error("Email already registered")

                    elif db.register_user(username, email, password, question, answer):
                        st.success("🎉 Account created successfully!")

                        # ✅ UPDATED MODERN EMAIL DESIGN
                        try:
                            welcome_subject = "🎉 Welcome to TextMorph!"

                            welcome_body = f"""
                            <!DOCTYPE html>
                            <html>
                            <head>
                                <style>
                                    body {{
                                        margin:0;
                                        padding:0;
                                        background:#0e1117;
                                        font-family:Arial;
                                    }}
                                    .container {{
                                        display:flex;
                                        justify-content:center;
                                        padding:40px;
                                    }}
                                    .card {{
                                        background:linear-gradient(145deg,#0F1B2D,#0C1828);
                                        border:1px solid rgba(0,200,255,0.3);
                                        border-radius:16px;
                                        padding:30px;
                                        width:420px;
                                        text-align:center;
                                        box-shadow:0 0 25px rgba(0,200,255,0.25);
                                    }}
                                    .title {{
                                        color:#00C8FF;
                                        font-size:22px;
                                        font-weight:bold;
                                    }}
                                    .text {{
                                        color:#b0c4de;
                                        margin:15px 0;
                                    }}
                                    .highlight {{
                                        color:#00FFC3;
                                        font-weight:bold;
                                        margin:10px 0;
                                    }}
                                    .btn {{
                                        display:inline-block;
                                        margin-top:20px;
                                        padding:12px 20px;
                                        background:linear-gradient(90deg,#00C8FF,#00FFC3);
                                        color:black;
                                        text-decoration:none;
                                        border-radius:8px;
                                        font-weight:bold;
                                    }}
                                </style>
                            </head>

                            <body>
                                <div class="container">
                                    <div class="card">

                                        <div class="title">🚀 Welcome, {username}!</div>

                                        <div class="text">
                                            Your account has been successfully created.
                                        </div>

                                        <div class="highlight">
                                            ✨ Summarize <br>
                                            🔁 Paraphrase <br>
                                            📊 Analyze Readability
                                        </div>

                                        <div class="text">
                                            Start exploring now!
                                        </div>

                                        <a href="https://infosysspringboard.ausnz.onwingspan.com/web/en/login"
                                           class="btn">
                                           🔑 Login Now
                                        </a>

                                    </div>
                                </div>
                            </body>
                            </html>
                            """

                            msg = MIMEMultipart()
                            msg['From'] = f"TextMorph <{EMAIL_ADDRESS}>"
                            msg['To'] = email
                            msg['Subject'] = welcome_subject
                            msg.attach(MIMEText(welcome_body, 'html'))

                            s = smtplib.SMTP('smtp.gmail.com', 587)
                            s.starttls()
                            s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
                            s.sendmail(EMAIL_ADDRESS, email, msg.as_string())
                            s.quit()

                            st.info(f"📧 Welcome email sent to {email}")

                        except Exception as e:
                            st.error(f"Email failed: {e}")

                        switch_page("login")

                    else:
                        st.error("User already exists.")

        if st.button("🔑 Go to Login", use_container_width=True):
            st.session_state.page = "login"
            st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)


def is_admin(email):
    if email == "admin@textmorph.com":
        return True

    conn = sqlite3.connect(db.DB_NAME)
    role = conn.execute(
        "SELECT role FROM user_roles WHERE email=?",
        (email,)
    ).fetchone()
    conn.close()



    if role and role[0].lower() == "admin":
        return True

    return False

    if role and role[0] == "admin":
        return True

    return False

# ---------------- LOGIN ----------------
def login():
    _auth_header()

    _, col, _ = st.columns([1, 1.4, 1], gap="medium")
    with col:

        st.markdown('<div class="auth-wrap">', unsafe_allow_html=True)

        with st.form("login_form"):

            st.markdown(f"""
            <div style="margin-bottom:20px">
                <div style="font-family:'Syne',sans-serif;font-size:1.4rem;font-weight:800;
                          color:{TEXT};">👋 Welcome back</div>
                <div style="font-size:.9rem;color:{MUTED};margin-top:6px;">
                    Sign in to continue
                </div>
            </div>
            """, unsafe_allow_html=True)

            email = st.text_input("Email")
            password = st.text_input("Password", type="password")
            submit = st.form_submit_button("Login")

            if submit:
                auth = db.authenticate_user(email, password)


                if auth == "locked":
                    st.error("Account locked. Try again later.")

                elif auth:

                    username = db.get_username(email)

                    st.session_state['user'] = email

                    st.session_state.token = create_token({
                        "email": email,
                        "username": username
                    })

                    st.success("Login Successful! 🎉")
                    time.sleep(1)


                    # Admin check
                    if is_admin(email):
                        st.session_state.page = "admin_dashboard"
                    else:
                        st.session_state.page = "dashboard"

                    st.rerun()

                else:
                    # Invalid login
                    st.error("Invalid credentials.")

                    # Check if old password used
                    old_dt = db.check_is_old_password(email, password)
                    if old_dt:
                        st.warning(f"Note: You used an old password from {get_relative_time(old_dt)}.")

        st.markdown('</div>', unsafe_allow_html=True)

        c1, c2 = st.columns(2, gap="medium")

        with c1:
            if st.button("Forgot password?", use_container_width=True):
                st.session_state.page = "forgot"
                st.rerun()

        with c2:
            if st.button("Create account", use_container_width=True):
                st.session_state.page = "signup"
                st.rerun()
# ---------------- DASHBOARD ----------------
def home_page():

    # ── Centred layout CSS — no fixed margins, no sidebar offset ──
    conn = sqlite3.connect(db.DB_NAME)
    st.markdown(f"""
    <style>
    /* Reset Streamlit padding so our grid can truly centre */

    [data-testid="block-container"] {{

        padding-top: 1rem;
    }}
    /* home-card grid handled by st.button CSS override below */
    /* Stats row */
    .stats-row {{
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        gap: 16px;
        width: 100%;
        margin-top: 4px;
    }}
    @media (max-width: 680px) {{
        .stats-row {{ grid-template-columns: repeat(2, 1fr); }}
    }}
    .stat-box {{
        background: rgba(13,24,46,0.95);
        border: 1px solid rgba(0,200,255,0.28);
        border-radius: 14px;
        padding: 18px 14px;
        text-align: center;
    }}
    .stat-num   {{ font-family:'Syne',sans-serif; font-size:1.8rem; font-weight:800; color:#00C8FF; }}
    .stat-label {{ font-size:.78rem; color:#6B8BA4; margin-top:4px; }}
    </style>""", unsafe_allow_html=True)

    # ── Page header ───────────────────────────────────────────
    st.markdown("""
    <div style="display:flex;align-items:center;gap:14px;margin-bottom:28px">
      <span style="font-size:2.2rem">🏠</span>
      <div>
        <p style="font-family:'Syne',sans-serif;font-weight:800;font-size:1.7rem;
                  color:#00C8FF;margin:0;line-height:1">Welcome back!</p>
        <p style="font-size:.85rem;color:#6B8BA4;margin:4px 0 0">
          What would you like to do today?</p>
      </div>
    </div>""", unsafe_allow_html=True)

    TOOLS = [
        ("📝", "Summarize",    "Condense long texts into crisp summaries with adjustable length and language output."),
        ("🔄", "Paraphrase",   "Rewrite content with different styles and complexity levels — from formal to creative."),
        ("📖", "Readability",  "Analyse your text across 5 readability indices and get actionable grade-level insights."),
        ("🗃️", "Augmentation", "Generate NLP training pairs at scale — ideal for fine-tuning your own models."),
        ("📜", "History",      "Browse your full activity log with filters, analytics, and CSV export."),
        ("👤", "Profile",      "Update your avatar, email, and password."),
    ]

    # ── Card grid: each card is a Streamlit button styled as a card ──
    st.markdown("""
    <style>
    /* Make each tool button look like a card — hide default button style */
    div[data-testid="stButton"] > button {
        background: linear-gradient(145deg, #0F1B2D, #0C1828) !important;
        border: 1px solid rgba(0,200,255,0.28) !important;
        border-radius: 18px !important;
        padding: 32px 20px 28px !important;
        min-height: 200px !important;
        display: flex !important;
        flex-direction: column !important;
        align-items: center !important;
        justify-content: center !important;
        gap: 10px !important;
        color: #E8F4FD !important;
        font-size: 0.84rem !important;
        font-family: 'DM Sans', sans-serif !important;
        white-space: normal !important;
        line-height: 1.5 !important;
        text-align: center !important;
        box-shadow: none !important;
        transition: all 0.25s ease !important;
    }

    /* Hover effect */
    div[data-testid="stButton"] > button:hover {
        border-color: #00C8FF !important;
        transform: translateY(-5px) !important;
        box-shadow: 0 10px 32px rgba(0,200,255,0.28) !important;
        background: linear-gradient(145deg, #112240, #0F1B2D) !important;
    }
    </style>
    """, unsafe_allow_html=True)

    # Row 1
    row1 = st.columns(3, gap="large")
    for col, (icon, title, desc) in zip(row1, TOOLS[:3]):
        with col:
            if st.button(
                f"{icon}\n\n**{title}**\n\n{desc}",
                key=f"home_btn_{title}",
                use_container_width=True
            ):
                st.session_state["_nav_to"] = title
                st.session_state["_active_page"] = title
                st.rerun()

    st.markdown("<div style='height:4px'></div>", unsafe_allow_html=True)

    # Row 2
    row2 = st.columns(3, gap="large")
    for col, (icon, title, desc) in zip(row2, TOOLS[3:]):
        with col:
            if st.button(
                f"{icon}\n\n**{title}**\n\n{desc}",
                key=f"home_btn_{title}",
                use_container_width=True
            ):
                st.session_state["_nav_to"] = title
                st.session_state["_active_page"] = title
                st.rerun()
    conn.close()

    st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)

    # ── Quick stats ───────────────────────────────────────────
    email = st.session_state["user"]
    acts  = db.get_user_activity(email) or []
    df_a  = pd.DataFrame(acts, columns=["Activity Type","Details","Output","Model Used","Timestamp"]) if acts else pd.DataFrame()

    total   = len(df_a)
    sums    = len(df_a[df_a["Activity Type"]=="Summarization"]) if len(df_a) else 0
    paras   = len(df_a[df_a["Activity Type"]=="Paraphrasing"])  if len(df_a) else 0
    models  = df_a["Model Used"].nunique() if len(df_a) else 0

    st.markdown(f"""
    <div class="stats-row">
      <div class="stat-box"><div class="stat-num">{total}</div><div class="stat-label">Total Operations</div></div>
      <div class="stat-box"><div class="stat-num">{sums}</div><div class="stat-label">Summarizations</div></div>
      <div class="stat-box"><div class="stat-num">{paras}</div><div class="stat-label">Paraphrases</div></div>
      <div class="stat-box"><div class="stat-num">{models}</div><div class="stat-label">Models Used</div></div>
    </div>""", unsafe_allow_html=True)

def user_profile(email):
    col1, col2 = st.columns([6,1])
    with col2:
        if st.button("⬅ Back", key="back_profile"):
            st.session_state["_nav_to"] = "Home"
            st.rerun()

    st.title("👤 User Profile")

    conn = sqlite3.connect(db.DB_NAME)

    # ---------------- CHANGE EMAIL ----------------

    st.subheader("📧 Change Email")

    new_email = st.text_input("Enter New Email")

    if st.button("Update Email"):

        if not new_email:
            st.error("Email cannot be empty")

        elif new_email == email:
            st.error("New email cannot be the same as the current email")

        elif not valid_email(new_email):
            st.error("Invalid email format")

        else:

            # check if email already exists
            existing = conn.execute(
                "SELECT email FROM users WHERE email=?",
                (new_email,)
            ).fetchone()

            if existing:
                st.error("Email already exists. Please use another email.")
            else:
                try:

                    conn.execute(
                        "UPDATE users SET email=? WHERE email=?",
                        (new_email, email)
                    )

                    conn.execute(
                        "UPDATE user_roles SET email=? WHERE email=?",
                        (new_email, email)
                    )

                    conn.execute(
                        "UPDATE activity_history SET email=? WHERE email=?",
                        (new_email, email)
                    )

                    conn.execute(
                        "UPDATE feedback SET email=? WHERE email=?",
                        (new_email, email)
                    )

                    conn.execute(
                        "UPDATE user_profiles SET email=? WHERE email=?",
                        (new_email, email)
                    )

                    conn.commit()

                    st.session_state.user = new_email

                    st.success("Email updated successfully")

                    st.rerun()

                except Exception as e:

                    conn.rollback()
                    st.error("Error updating email")

    st.markdown("---")


    # ---------------- CHANGE PASSWORD ----------------

    st.subheader("🔑 Change Password")

    new_password = st.text_input("New Password", type="password")
    confirm_password = st.text_input("Confirm Password", type="password")

    if st.button("Update Password"):

        if not new_password or not confirm_password:
            st.error("Both password fields are required")

        elif not valid_password(new_password):
            st.error("Password must contain uppercase, lowercase, number and special character")

        elif new_password != confirm_password:
            st.error("Passwords do not match")

        else:
            hashed = bcrypt.hashpw(
                new_password.encode(),
                bcrypt.gensalt()
            )

            conn.execute(
                "UPDATE users SET password=? WHERE email=?",
                (hashed, email)
            )

            conn.commit()

            st.success("Password updated successfully")


    st.markdown("---")


    # ---------------- AVATAR UPLOAD ----------------

    st.subheader("🖼 Upload Avatar")

    avatar = st.file_uploader(
        "Upload Profile Picture",
        type=["png","jpg","jpeg"]
    )

    if avatar:

        img = avatar.read()

        conn.execute(
            "REPLACE INTO user_profiles(email,avatar) VALUES (?,?)",
            (email,img)
        )

        conn.commit()

        st.success("Avatar Updated")

        st.rerun()


    # ---------------- SHOW AVATAR ----------------

    data = conn.execute(
        "SELECT avatar FROM user_profiles WHERE email=?",
        (email,)
    ).fetchone()

    if data and data[0]:
        st.image(data[0], width=150)


    # ---------------- DELETE AVATAR ----------------

    if st.button("Delete Profile Picture"):
        db.delete_profile_image(email)
        st.success("Profile picture deleted!")
        st.rerun()


# ---------------- FORGOT PASSWORD ----------------
def forgot_password():
      # Initialize session state (FIX)
    if "stage" not in st.session_state:
      st.session_state.stage = "email"

    if "reset_email" not in st.session_state:
        st.session_state.reset_email = ""

    if "otp_token" not in st.session_state:
        st.session_state.otp_token = None

    if "otp_sent_time" not in st.session_state:
        st.session_state.otp_sent_time = None

    st.title("🔒 Forgot Password")

    # -------- STAGE 1: EMAIL --------
    if st.session_state.stage == "email":
        email = st.text_input("Enter your registered Email")

        if st.button("Verify Email"):
            if db.check_user_exists(email):
                st.session_state.reset_email = email
                st.session_state.email_verified = True
                st.success("Email Verified ✅")
            else:
                st.error("Email not found")

        # Show methods on SAME PAGE after verification
        if st.session_state.get("email_verified"):

            st.markdown("---")
            st.subheader("Choose Reset Method")

            col1, space, col2 = st.columns([1.3, 1, 1.3])

            with col1:
                if st.button("Reset via OTP", use_container_width=True):
                    st.session_state.stage = "otp"
                    st.rerun()

            with col2:
                if st.button("Reset via Security Question", use_container_width=True):
                    st.session_state.stage = "security"
                    st.rerun()
        st.markdown("<br><br>", unsafe_allow_html=True)
    # -------- STAGE 3A: OTP --------
    elif st.session_state.stage == "otp":
        st.subheader("OTP Verification")
        st.info(f"OTP will be sent to {st.session_state.reset_email}")

        OTP_VALID_SECONDS = 600

        # ---------- SEND OTP ----------
        if not st.session_state.otp_sent_time:
            if st.button("Send OTP"):
                otp = generate_otp()
                ok, msg = send_email(
                    st.session_state.reset_email,
                    otp,
                    EMAIL_PASSWORD
                )

                if ok:
                    st.session_state.otp_token = create_otp_token(
                        otp,
                        st.session_state.reset_email
                    )
                    st.session_state.otp_sent_time = time.time()
                    st.success("OTP sent successfully 📧")
                else:
                    st.error(f"Failed to send OTP: {msg}")

        # ---------- AFTER OTP SENT ----------
        if st.session_state.otp_sent_time:
            elapsed = time.time() - st.session_state.otp_sent_time
            remaining = int(OTP_VALID_SECONDS - elapsed)

            if remaining <= 0:
                st.error("OTP expired. Please resend.")
                st.session_state.otp_sent_time = None
                st.session_state.otp_token = None
            else:
                st.info(f"OTP expires in {remaining} seconds")

                otp_input = st.text_input("Enter OTP")

                col1, space, col2 = st.columns([1,2,1])

                # Verify
                if col1.button("Verify OTP",use_container_width=True):
                    if not otp_input.strip():
                        st.error("Please enter OTP")
                    else:
                        ok, msg = verify_otp_token(
                            st.session_state.otp_token,
                            otp_input.strip(),
                            st.session_state.reset_email
                        )
                        if ok:
                            st.success("OTP Verified ✅")
                            st.session_state.stage = "reset"
                            st.session_state.otp_sent_time = None
                            st.rerun()
                        else:
                            st.error("Invalid OTP")

                # Resend
                if col2.button("Resend OTP",use_container_width=True):
                    st.session_state.otp_sent_time = None
                    st.session_state.otp_token = None
                    st.rerun()

    # -------- STAGE 3B: SECURITY QUESTION --------
    elif st.session_state.stage == "security":
        question = db.get_security_question(st.session_state.reset_email)
        st.write(f"Security Question: {question}")

        answer = st.text_input("Enter Answer")

        if st.button("Verify Answer"):
            if db.verify_security_answer(
                st.session_state.reset_email,
                answer
            ):
                st.session_state.stage = "reset"
                st.success("Answer Verified ✅")
                st.rerun()
            else:
                st.error("Incorrect Answer")

    # -------- STAGE 4: RESET PASSWORD --------
    elif st.session_state.stage == "reset":
        new_pass = st.text_input("New Password", type="password")
        confirm_pass = st.text_input("Confirm Password", type="password")

        if st.button("Update Password"):
            if not valid_password(new_pass):
                st.error("Weak password")
            elif new_pass != confirm_pass:
                st.error("Passwords do not match")
            elif db.check_password_reused(
                st.session_state.reset_email,
                new_pass
            ):
                st.error("You cannot reuse old password")
            else:
                db.update_password(
                    st.session_state.reset_email,
                    new_pass
                )
                st.success("Password Updated Successfully 🎉")

    if st.button("Back to Login"):
        st.session_state.stage = "email"
        st.session_state.page = "login"
        st.rerun()


    st.markdown('</div>', unsafe_allow_html=True)
def _clear_stale_results(selected):
    pass
# ---------------- ROUTING ----------------
if "user" not in st.session_state:
    st.session_state.user = None

if "page" not in st.session_state:
    st.session_state.page = "login"


# ------------------ MAIN ROUTER ------------------
import base64

if st.session_state.user:

    # ✅ FORCE SIDEBAR OPEN
    st.session_state.sidebar_state = "expanded"

    # -------- SIDEBAR --------
    with st.sidebar:

        st.image(
            "https://www.infosys.com/content/dam/infosys-web/en/about/springboard/images/infosys-springboard.png",
            width=150
        )

        try:
            payload = jwt.decode(
                st.session_state.token,
                SECRET_KEY,
                algorithms=[ALGORITHM]
            )
        except:
            st.session_state.clear()
            st.session_state.page = "login"
            st.rerun()

        email = st.session_state["user"]
        username = db.get_username(email)
        avatar = db.get_profile_image(email)

        if avatar:
            img_base64 = base64.b64encode(avatar).decode()
            st.markdown(f"""
            <div style="display:flex; justify-content:flex-start;">
                <img src="data:image/png;base64,{img_base64}"
                style="width:100px;height:100px;border-radius:50%;
                object-fit:cover;border:3px solid #00f5ff;">
            </div>
            """, unsafe_allow_html=True)
        else:
            st.image(
                "https://cdn-icons-png.flaticon.com/512/149/149071.png",
                width=80
            )

        greeting = get_greeting()

        st.markdown(f"""
        <div style="
            font-size:20px;
            font-weight:bold;
            color:#6B8BA4;
            margin-bottom:8px;   /* 👈 space below username */
        ">
            👤 {username}
        </div>

        <div style="
            font-size:14px;
            color:#B8E4F9;
            margin-top:6px;     /* 👈 space above greeting */
        ">
            {greeting}
        </div>
        """, unsafe_allow_html=True)

        st.markdown("---")

        if st.button("🔓 Logout", key="logout_btn", use_container_width=False):
            st.session_state.clear()
            st.session_state.page = "login"
            st.rerun()

    # -------- NAVIGATION --------
    if "_nav_to" not in st.session_state:
        st.session_state["_nav_to"] = "Home"

    page = st.session_state["_nav_to"]
    email = st.session_state["user"]

    is_admin_user = is_admin(email)

    # ================= USER =================
    if not is_admin_user:

        if page == "Home":
            home_page()

        elif page == "Summarize":
            summarizer_page()

        elif page == "Paraphrase":
            paraphraser_page()

        elif page == "Readability":
            readability_page()

        elif page == "Augmentation":
            augmentation_page()

        elif page == "History":
            history_page()

        elif page == "Profile":
            user_profile(email)

    # ================= ADMIN =================
    else:

        if page == "Home":
            admin_home_page()

        elif page == "Users":
            user_management()

        elif page == "Analytics":
            analytics_dashboard()

        elif page == "Activity":
            activity_tracking()

        elif page == "Remove Admin":
            remove_admin()

        elif page == "Feedback":
            feedback_section()

        elif page == "Locked":
            locked_accounts_section()

        elif page == "Download":
            export_data()

# ------------------ NOT LOGGED IN ------------------
else:

    if st.session_state.page == "login":
        login()

    elif st.session_state.page == "signup":
        signup()

    elif st.session_state.page == "forgot":
        forgot_password()

Writing app.py


In [ ]:
import os
import subprocess
import time
from google.colab import userdata
from pyngrok import ngrok

# 1. Retrieve secrets safely in the kernel
email_pass = None
ngrok_token = None

try:
    try:
        email_pass = userdata.get('EMAIL_PASSWORD')
    except Exception as e:
        print(f"⚠️ Warning: EMAIL_PASSWORD secret not found: {e}")

    try:
        ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    except Exception as e:
        print(f"⚠️ Warning: NGROK_AUTHTOKEN secret not found: {e}")

    # Store in environment for the subprocess to see
    if email_pass:
        os.environ['EMAIL_PASSWORD'] = email_pass
    os.environ['JWT_SECRET'] = "super-secret-change-me"

except Exception as e:
    print(f"❌ Error setting up environment: {e}")

# 2. Authenticate Ngrok
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

    # 3. Kill old processes to prevent conflicts
    ngrok.kill()
    time.sleep(1)

    # 4. Run Streamlit in the background
    # We pass os.environ so it inherits the secrets we just set
    process = subprocess.Popen(['streamlit', 'run', 'app.py'], env=os.environ.copy())

    # Give it a moment to start
    time.sleep(3)

    # 5. Open Tunnel
    try:
        public_url = ngrok.connect(8501).public_url
        print(f"\n🚀 Your App is running at: {public_url}")
        print("\n👇 Click the link above to open the app!")
    except Exception as e:
        print(f"❌ Error starting Ngrok tunnel: {e}")

    # 6. Keep Alive / Manual Stop
    print("\n✅ App is running! Check the URL above.")
    try:
        input("\n🛑 Press ENTER in this box to STOP the server...\n")
    except (KeyboardInterrupt, EOFError):
        pass
    finally:
        process.terminate()
        ngrok.kill()
        print("✅ Server and Tunnel stopped.")

else:
    print("❌ No Ngrok Token found. Please add 'NGROK_AUTHTOKEN' to Colab Secrets.")



🚀 Your App is running at: https://postrorse-unstatesmanlike-deloras.ngrok-free.dev

👇 Click the link above to open the app!

✅ App is running! Check the URL above.
